In [ ]:
import os
import random
import glob
import re

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

import itertools
from datetime import timedelta
from tqdm import tqdm

seed = 777
LOOKBACK, PREDICT, BATCH_SIZE, EPOCHS = 28, 7, 16, 50
DROP_COUNT = 17

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(seed)

train = pd.read_csv('train/train.csv')
holiday = pd.to_datetime([
    '2023-01-01', '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24', '2023-03-01', '2023-05-05', '2023-05-27', '2023-05-29', '2023-06-06', '2023-08-15', '2023-09-28', '2023-09-29', '2023-09-30', '2023-10-01', '2023-10-02', '2023-10-03', '2023-10-09', '2023-12-25',
    '2024-01-01', '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12', '2024-03-01', '2024-04-10', '2024-05-06', '2024-05-15', '2024-06-06', '2024-08-15', '2024-09-16', '2024-09-17', '2024-09-18', '2024-10-01', '2024-10-03', '2024-10-09', '2024-12-25',
    '2025-01-01', '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30', '2025-03-03', '2025-05-05', '2025-05-06', '2025-06-03', '2025-06-06', '2025-08-15',
])


def check_holiday(date: pd.Timestamp) -> bool:
    return date.dayofweek in (5, 6) or date in holiday


def filter_long_zero_runs(X, y):
    lag_cols = [f'lag_{i}' for i in range(1, LOOKBACK+1)]
    def _max_zero_run(arr):
        return max((len(list(g)) for v, g in itertools.groupby(arr) if v == 0.0), default=0)
    X['max_zero_run'] = X[lag_cols].apply(_max_zero_run, axis=1)
    mask = X['max_zero_run'] < DROP_COUNT
    return X.loc[mask].drop(columns='max_zero_run'), y.loc[mask]


def interpolate_single_zero_lags(X):
    lag_cols = [f'lag_{i}' for i in range(1, LOOKBACK+1)]
    arr = X[lag_cols].astype(float).to_numpy(copy=True)
    L = arr.shape[1]
    mask = np.zeros_like(arr, dtype=bool)

    for j in range(1, L-1):
        mask[:, j] = (arr[:, j]==0) & (arr[:, j-1]!=0) & (arr[:, j+1]!=0)

    arr[mask] = np.nan
    df_interp = pd.DataFrame(arr, columns=lag_cols, index=X.index).interpolate(axis=1, method='linear').fillna(0)
    X[lag_cols] = df_interp
    return X


def preprocess(df, is_train=True):
    df.loc[df['매출수량'] < 0, '매출수량'] = 0
    df['영업일자'] = pd.to_datetime(df['영업일자'])
    df['dow'] = df['영업일자'].dt.dayofweek
    df['month'] = df['영업일자'].dt.month
    df['holiday'] = df['영업일자'].map(check_holiday).astype(int)

    def make_feature_row(key, ref_date, lag_block):
        data = {
            '영업장명_메뉴명': key,
            'dow': ref_date.dayofweek,
            'month': ref_date.month,
            'holiday': int(check_holiday(ref_date)),
            'sin_day': np.sin(2*np.pi*ref_date.timetuple().tm_yday/365.0),
            'cos_day': np.cos(2*np.pi*ref_date.timetuple().tm_yday/365.0),
            'ref_date': ref_date,
        }
        data.update({f'holiday_plus_{d}d': int(check_holiday(ref_date + pd.Timedelta(days=d))) for d in range(1, 8)})
        data.update({f'lag_{j+1}': float(val) for j, val in enumerate(lag_block)})
        return data


    X_data, y_data = [], []
    for key, group in df.groupby('영업장명_메뉴명'):
        group = group.sort_values('영업일자')
        sales, dates = group['매출수량'].to_numpy(), group['영업일자']

        if len(sales) < LOOKBACK:
            continue

        if is_train:
            Xy = [
                (make_feature_row(key, dates.iloc[i], sales[i-LOOKBACK:i][::-1]), sales[i:i+PREDICT])
                for i in range(LOOKBACK, len(sales) - PREDICT + 1)
            ]
            X_data.extend([r for r, _ in Xy])
            y_data.extend([y for _, y in Xy])
        else:
            ref_date = dates.iloc[-1] + timedelta(days=1)
            lag_block = sales[-LOOKBACK:][::-1]
            X_data.append(make_feature_row(key, ref_date, lag_block))

    X = pd.DataFrame.from_records(X_data)
    y = pd.DataFrame(y_data, columns=[f'target_{i}' for i in range(1, PREDICT+1)]) if is_train else None

    if is_train:
        X, y = filter_long_zero_runs(X, y)

    X = interpolate_single_zero_lags(X)

    # special_stores = {'느티나무 BBQ','라그로타','화담숲주막','화담숲카페'}
    # X['holiday2'] = ((X['영업장명_메뉴명'].str.split('_').str[0].isin(special_stores)) & (X['dow']==0)).astype(int)

    return X, y

def round_and_clip_min1(arr):
    a = np.rint(np.asarray(arr, dtype=float))
    a = np.where(a < 1.0, 1.0, a)
    return a

x_train, y_train = preprocess(train, is_train=True)

from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

onehot = OneHotEncoder(handle_unknown='ignore')

categorical_features = ['영업장명_메뉴명', 'dow', 'month']
numerical_features = ['holiday'] + x_train.filter(regex='holiday_plus').columns.tolist() + ['sin_day', 'cos_day'] + [f'lag_{l}' for l in range(1, LOOKBACK + 1)]

x_cat_train = onehot.fit_transform(x_train[categorical_features]).toarray()
x_num_train = x_train[numerical_features].values
x_train_processed = np.hstack([x_cat_train, x_num_train])

model_params = {
    'n_estimators': 1000,
    'subsample': 0.8,
    'max_depth': 6,
    'colsample_bytree': 0.8,
    'learning_rate': 0.05,
    'objective': 'reg:squarederror',
    'random_state': seed,
}
model = XGBRegressor(**model_params)
multi_output_regressor = MultiOutputRegressor(model)

multi_output_regressor.fit(x_train_processed, y_train)

all_preds = []

test_files = sorted(glob.glob('test/TEST_*.csv'))
for path in test_files:
    results = []

    test_df = pd.read_csv(path)
    x_test, _ = preprocess(test_df, is_train=False)

    x_cat_test = onehot.transform(x_test[categorical_features]).toarray()
    x_num_test = x_test[numerical_features].values
    x_test_processed = np.hstack([x_cat_test, x_num_test])

    preds = multi_output_regressor.predict(x_test_processed)

    for i in range(len(x_test)):
        menu = x_test.iloc[i]['영업장명_메뉴명']

        for j in range(PREDICT):
            date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
            results.append({
                '영업일자': date,
                '영업장명_메뉴명': menu,
                '매출수량': preds[i][j]
            })

    pred_df = pd.DataFrame(results)
    all_preds.append(pred_df)

full_pred_df = pd.concat(all_preds, ignore_index=True)

from collections import defaultdict

# 기존 전체 모델 예측 결과 저장
all_preds_global = full_pred_df.copy()

# 개별 모델 예측 결과 저장
all_preds_local = []

unique_keys = train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).unique()

for key in tqdm(unique_keys):
    x_train_key, y_train_key = x_train[x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key], y_train[x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key]

    # 카테고리 인코딩 (key만 존재하므로 카테고리 의미 없음)
    x_cat_train_key = onehot.fit_transform(x_train_key[categorical_features]).toarray()
    x_num_train_key = x_train_key[numerical_features].values
    x_train_processed_key = np.hstack([x_cat_train_key, x_num_train_key])

    # 개별 모델 학습
    model_key = XGBRegressor(**model_params)
    multi_reg_key = MultiOutputRegressor(model_key)
    multi_reg_key.fit(x_train_processed_key, y_train_key)

    # 각 test 파일별 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)

        x_test_key = x_test[x_test['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key]

        x_cat_test_key = onehot.transform(x_test_key[categorical_features]).toarray()
        x_num_test_key = x_test_key[numerical_features].values
        x_test_processed_key = np.hstack([x_cat_test_key, x_num_test_key])

        preds_key = multi_reg_key.predict(x_test_processed_key)

        results_key = []
        for i in range(len(x_test_key)):
            menu = x_test_key.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_key.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_key[i][j]
                })

        pred_df_key = pd.DataFrame(results_key)
        all_preds_local.append(pred_df_key)

# 개별 모델 전체 결과 합치기
full_pred_df_local = pd.concat(all_preds_local, ignore_index=True)
all_preds_local_menu = []
unique_store_menu = train['영업장명_메뉴명'].unique()

for sm in tqdm(unique_store_menu):
    y_train_sm = y_train[x_train['영업장명_메뉴명'] == sm]
    x_train_sm = x_train[x_train['영업장명_메뉴명'] == sm]

    # 카테고리 / 수치 분리
    x_cat_sm = onehot.fit_transform(x_train_sm[categorical_features]).toarray()
    x_num_sm = x_train_sm[numerical_features].values
    x_train_processed_sm = np.hstack([x_cat_sm, x_num_sm])

    model_sm = XGBRegressor(**model_params)
    multi_reg_sm = MultiOutputRegressor(model_sm)
    multi_reg_sm.fit(x_train_processed_sm, y_train_sm)

    # 테스트 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)
        x_test_sm = x_test[x_test['영업장명_메뉴명'] == sm]

        if len(x_test_sm) == 0:
            continue  # 해당 메뉴 없는 경우 건너뛰기

        x_cat_test_sm = onehot.transform(x_test_sm[categorical_features]).toarray()
        x_num_test_sm = x_test_sm[numerical_features].values
        x_test_processed_sm = np.hstack([x_cat_test_sm, x_num_test_sm])

        preds_sm = multi_reg_sm.predict(x_test_processed_sm)

        results_sm = []
        for i in range(len(x_test_sm)):
            menu = x_test_sm.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_sm.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_sm[i][j]
                })
        all_preds_local_menu.append(pd.DataFrame(results_sm))

full_pred_df_menu = pd.concat(all_preds_local_menu, ignore_index=True)

# group = {
#     0: ['담하', '미라시아', '느티나무 셀프BBQ', '포레스트릿', '카페테리아'],
#     1: ['라그로타'],
#     2: ['연회장'],
#     3: ['화담숲주막', '화담숲카페']
# }

group = {
    0: ['담하', '미라시아', '느티나무 셀프BBQ', '포레스트릿'],
    1: ['라그로타', '카페테리아'],
    2: ['연회장'],
    3: ['화담숲주막', '화담숲카페']
}

all_preds_loc = []

for group_id, store_list in tqdm(group.items()):
    # 해당 그룹에 속하는 train 데이터 선택
    mask_train = x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).isin(store_list)
    x_train_group = x_train[mask_train]
    y_train_group = y_train[mask_train]

    # 카테고리 인코딩
    x_cat_train_group = onehot.fit_transform(x_train_group[categorical_features]).toarray()
    x_num_train_group = x_train_group[numerical_features].values
    x_train_processed_group = np.hstack([x_cat_train_group, x_num_train_group])

    # 그룹별 모델 학습
    model_group = XGBRegressor(**model_params)
    multi_reg_group = MultiOutputRegressor(model_group)
    multi_reg_group.fit(x_train_processed_group, y_train_group)

    # 각 test 파일별 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)

        # 해당 그룹 test 데이터 선택
        mask_test = x_test['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).isin(store_list)
        x_test_group = x_test[mask_test]

        if len(x_test_group) == 0:
            continue  # 해당 그룹 데이터가 없으면 스킵

        x_cat_test_group = onehot.transform(x_test_group[categorical_features]).toarray()
        x_num_test_group = x_test_group[numerical_features].values
        x_test_processed_group = np.hstack([x_cat_test_group, x_num_test_group])

        preds_group = multi_reg_group.predict(x_test_processed_group)

        results_group = []
        for i in range(len(x_test_group)):
            menu = x_test_group.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_group.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_group[i][j]
                })

        pred_df_group = pd.DataFrame(results_group)
        all_preds_loc.append(pred_df_group)

# 그룹별 모델 전체 결과 합치기
full_pred_df_loc = pd.concat(all_preds_loc, ignore_index=True)

menu_groups = {
    "느티나무 셀프BBQ": {
        "메인메뉴": [
            "느티나무 셀프BBQ_BBQ55(단체)",
            "느티나무 셀프BBQ_본삼겹 (단품,실내)",
            "느티나무 셀프BBQ_신라면",
            "느티나무 셀프BBQ_육개장 사발면",
            "느티나무 셀프BBQ_햇반"
        ],
        "사이드": [
            "느티나무 셀프BBQ_쌈야채세트",
            "느티나무 셀프BBQ_쌈장",
            "느티나무 셀프BBQ_허브솔트"
        ],
        "음료": [
            "느티나무 셀프BBQ_참이슬 (단체)",
            "느티나무 셀프BBQ_카스 병(단체)",
            "느티나무 셀프BBQ_콜라 (단체)",
            "느티나무 셀프BBQ_스프라이트 (단체)"
        ],
        "소모품": [
            "느티나무 셀프BBQ_1인 수저세트",
            "느티나무 셀프BBQ_일회용 소주컵",
            "느티나무 셀프BBQ_일회용 종이컵",
            "느티나무 셀프BBQ_친환경 접시 14cm",
            "느티나무 셀프BBQ_친환경 접시 23cm"
        ],
        "대여": [
            "느티나무 셀프BBQ_대여료 30,000원",
            "느티나무 셀프BBQ_대여료 60,000원",
            "느티나무 셀프BBQ_대여료 90,000원",
            "느티나무 셀프BBQ_잔디그늘집 대여료 (6인석)",
            "느티나무 셀프BBQ_잔디그늘집 대여료 (12인석)",
            "느티나무 셀프BBQ_잔디그늘집 의자 추가"
        ]
    },
    "담하": {
        "메인메뉴": [
            "담하_(단체) 생목살 김치전골 2.0",
            "담하_(단체) 은이버섯 갈비탕",
            "담하_(단체) 한우 우거지 국밥",
            "담하_(단체) 황태해장국 3/27까지",
            "담하_(정식) 된장찌개",
            "담하_(정식) 물냉면 ",
            "담하_(정식) 비빔냉면",
            "담하_(후식) 된장찌개",
            "담하_(후식) 물냉면",
            "담하_(후식) 비빔냉면",
            "담하_갑오징어 비빔밥",
            "담하_갱시기",
            "담하_꼬막 비빔밥",
            "담하_담하 한우 불고기",
            "담하_담하 한우 불고기 정식",
            "담하_더덕 한우 지짐",
            "담하_들깨 양지탕",
            "담하_명태회 비빔냉면",
            "담하_봉평메밀 물냉면",
            "담하_생목살 김치찌개",
            "담하_은이버섯 갈비탕",
            "담하_한우 떡갈비 정식",
            "담하_한우 미역국 정식",
            "담하_한우 우거지 국밥",
            "담하_한우 차돌박이 된장찌개",
            "담하_황태해장국"
        ],
        "사이드": [
            "담하_(단체) 공깃밥",
            "담하_공깃밥",
            "담하_라면사리",
            "담하_메밀면 사리"
        ],
        "음료": [
            "담하_느린마을 막걸리",
            "담하_명인안동소주",
            "담하_문막 복분자 칵테일",
            "담하_스프라이트",
            "담하_제로콜라",
            "담하_참이슬",
            "담하_처음처럼",
            "담하_카스",
            "담하_콜라",
            "담하_테라",
            "담하_하동 매실 칵테일"
        ],
        "대여": [
            "담하_룸 이용료"
        ]
    },
    "라그로타": {
        "메인메뉴": [
            "라그로타_AUS (200g)",
            "라그로타_한우 (200g)",
            "라그로타_양갈비 (4ps)",
            "라그로타_까르보나라",
            "라그로타_알리오 에 올리오 ",
            "라그로타_버섯 크림 리조또",
            "라그로타_해산물 토마토 리조또",
            "라그로타_해산물 토마토 스파게티",
            "라그로타_해산물 토마토 스튜 파스타",
            "라그로타_모둠 해산물 플래터"
        ],
        "샐러드·사이드": [
            "라그로타_그릴드 비프 샐러드",
            "라그로타_시저 샐러드 ",
            "라그로타_빵 추가 (1인)",
            "라그로타_Open Food"
        ],
        "음료": [
            "라그로타_아메리카노",
            "라그로타_자몽리치에이드",
            "라그로타_스프라이트",
            "라그로타_제로콜라",
            "라그로타_콜라",
            "라그로타_G-Charge(3)",
            "라그로타_Gls.Sileni",
            "라그로타_Gls.미션 서드",
            "라그로타_미션 서드 카베르네 쉬라",
            "라그로타_카스",
            "라그로타_하이네켄(생)"
        ],
    },
    "미라시아": {
        "브런치·패키지": [
            "미라시아_(단체)브런치주중 36,000",
            "미라시아_미라시아 브런치 (패키지)",
            "미라시아_브런치 2인 패키지 ",
            "미라시아_브런치 4인 패키지 ",
            "미라시아_브런치(대인) 주말",
            "미라시아_브런치(대인) 주중",
            "미라시아_브런치(어린이)"
        ],
        "플래터·피자·파스타": [
            "미라시아_(오븐) 하와이안 쉬림프 피자",
            "미라시아_(화덕) 불고기 페퍼로니 반반피자",
            "미라시아_BBQ Platter",
            "미라시아_BBQ 고기추가",
            "미라시아_보일링 랍스타 플래터",
            "미라시아_보일링 랍스타 플래터(덜매운맛)",
            "미라시아_쉬림프 투움바 파스타",
            "미라시아_오븐구이 윙과 킬바사소세지"
        ],
        "사이드·추가": [
            "미라시아_공깃밥",
            "미라시아_칠리 치즈 프라이",
            "미라시아_파스타면 추가(150g)",
            "미라시아_콥 샐러드"
        ],
        "음료": [
            "미라시아_스프라이트",
            "미라시아_애플망고 에이드",
            "미라시아_핑크레몬에이드",
            "미라시아_코카콜라",
            "미라시아_코카콜라(제로)",
            "미라시아_글라스와인 (레드)",
            "미라시아_레인보우칵테일(알코올)",
            "미라시아_버드와이저(무제한)",
            "미라시아_스텔라(무제한)",
            "미라시아_얼그레이 하이볼",
            "미라시아_유자 하이볼",
            "미라시아_잭 애플 토닉"
        ],
    },
    "연회장": {
        "공간대여": [
            "연회장_Conference L1",
            "연회장_Conference L2",
            "연회장_Conference L3",
            "연회장_Conference M1",
            "연회장_Conference M8",
            "연회장_Conference M9",
            "연회장_Convention Hall",
            "연회장_Grand Ballroom",
            "연회장_OPUS 2"
        ],
        "메인요리": [
            "연회장_돈목살 김치찌개 (밥포함)",
            "연회장_마라샹궈",
            "연회장_매콤 무뼈닭발&계란찜",
            "연회장_모둠 돈육구이(3인)",
            "연회장_왕갈비치킨"
        ],
        "기타": [
            "연회장_공깃밥",
            "연회장_삼겹살추가 (200g)",
            "연회장_야채추가",
            "연회장_주먹밥 (2ea)",
            "연회장_Cass Beer",
            "연회장_Regular Coffee",
            "연회장_Cookie Platter",
            "연회장_골뱅이무침",
            "연회장_로제 치즈떡볶이"
        ],
    },
    "카페테리아": {
        "정식·단체식": [
            "카페테리아_단체식 13000(신)",
            "카페테리아_단체식 18000(신)",
            "카페테리아_한상 삼겹구이 정식(2인) 소요시간 약 15~20분",
            "카페테리아_오픈푸드"
        ],
        "단품메뉴": [
            "카페테리아_돼지고기 김치찌개",
            "카페테리아_수제 등심 돈까스",
            "카페테리아_치즈돈까스",
            "카페테리아_어린이 돈까스",
            "카페테리아_약 고추장 돌솥비빔밥",
            "카페테리아_진사골 설렁탕"
        ],
        "면·밥류": [
            "카페테리아_새우 볶음밥",
            "카페테리아_새우튀김 우동",
            "카페테리아_짜장면",
            "카페테리아_짜장밥",
            "카페테리아_짬뽕",
            "카페테리아_짬뽕밥"
        ],
        "음료": [
            "카페테리아_아메리카노(HOT)",
            "카페테리아_아메리카노(ICE)",
            "카페테리아_카페라떼(HOT)",
            "카페테리아_카페라떼(ICE)",
            "카페테리아_복숭아 아이스티"
        ],
        "사이드·추가": [
            "카페테리아_공깃밥(추가)",
            "카페테리아_샷 추가",
            "카페테리아_구슬아이스크림"
        ]
    },
    "포레스트릿": {
        "분식": [
            "포레스트릿_떡볶이",
            "포레스트릿_꼬치어묵",
            "포레스트릿_치즈 핫도그",
            "포레스트릿_페스츄리 소시지"
        ],
        "음료": [
            "포레스트릿_아메리카노(HOT)",
            "포레스트릿_아메리카노(ICE)",
            "포레스트릿_카페라떼(HOT)",
            "포레스트릿_카페라떼(ICE)",
            "포레스트릿_복숭아 아이스티",
            "포레스트릿_코카콜라",
            "포레스트릿_스프라이트",
            "포레스트릿_생수"
        ]
    },
    "화담숲주막": {
        "음료": [
            "화담숲주막_느린마을 막걸리",
            "화담숲주막_참살이 막걸리",
            "화담숲주막_단호박 식혜 ",
            "화담숲주막_찹쌀식혜",
            "화담숲주막_콜라",
            "화담숲주막_스프라이트"
        ],
        "안주": [
            "화담숲주막_해물파전",
            "화담숲주막_병천순대"
        ]
    },
    "화담숲카페": {
        "전통 음료": [
            "화담숲카페_메밀미숫가루",
            "화담숲카페_현미뻥스크림"
        ],
        "커피": [
            "화담숲카페_아메리카노 HOT",
            "화담숲카페_아메리카노 ICE",
            "화담숲카페_카페라떼 ICE"
        ]
    }

}

all_preds_group = []

for store_name, groups in tqdm(menu_groups.items()):
    for group_name, menu_list in groups.items():
        # 그룹 데이터 추출
        x_train_group = x_train[x_train['영업장명_메뉴명'].isin(menu_list)]
        y_train_group = y_train.loc[x_train_group.index]

        if len(x_train_group) == 0:
            continue

        # 카테고리 인코딩
        x_cat_train_group = onehot.fit_transform(x_train_group[categorical_features]).toarray()
        x_num_train_group = x_train_group[numerical_features].values
        x_train_processed_group = np.hstack([x_cat_train_group, x_num_train_group])

        # 그룹별 모델 학습
        model_group = XGBRegressor(**model_params)
        multi_reg_group = MultiOutputRegressor(model_group)
        multi_reg_group.fit(x_train_processed_group, y_train_group)

        # 각 test 파일별 예측
        for path in test_files:
            test_df = pd.read_csv(path)
            x_test, _ = preprocess(test_df, is_train=False)

            x_test_group = x_test[x_test['영업장명_메뉴명'].isin(menu_list)]
            if len(x_test_group) == 0:
                continue

            x_cat_test_group = onehot.transform(x_test_group[categorical_features]).toarray()
            x_num_test_group = x_test_group[numerical_features].values
            x_test_processed_group = np.hstack([x_cat_test_group, x_num_test_group])

            preds_group = multi_reg_group.predict(x_test_processed_group)

            results_group = []
            for i in range(len(x_test_group)):
                menu = x_test_group.iloc[i]['영업장명_메뉴명']
                for j in range(PREDICT):
                    date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                    results_group.append({
                        '영업일자': date,
                        '영업장명_메뉴명': menu,
                        '매출수량': preds_group[i][j],
                        '그룹명': group_name   # 그룹 단위 정보 추가
                    })

            pred_df_group = pd.DataFrame(results_group)
            all_preds_group.append(pred_df_group)

full_pred_df_group = pd.concat(all_preds_group, ignore_index=True)

# === 앙상블 ===
ensemble_preds = all_preds_global.copy()

full_pred_df_local = full_pred_df_local.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_local = full_pred_df_local.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_menu = full_pred_df_menu.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_menu = full_pred_df_menu.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_loc = full_pred_df_loc.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_loc = full_pred_df_loc.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_group = full_pred_df_group.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_group = full_pred_df_group.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

ensemble_preds['매출수량'] = ensemble_preds['매출수량']*(1/5) + full_pred_df_local['매출수량']*(1/5) + full_pred_df_menu['매출수량']*(1/5) + full_pred_df_loc['매출수량']*(1/5) + full_pred_df_group['매출수량']*(1/5)

# 제출 파일 생성
submission = pd.read_csv('sample_submission.csv')
cols = submission.columns.drop('영업일자')
for col in cols:
    df = ensemble_preds[ensemble_preds['영업장명_메뉴명'] == col]
    df = df.set_index('영업일자')
    df = df.loc[submission['영업일자']].reset_index()
    submission[col] = round_and_clip_min1(df['매출수량'])

submission[cols] = submission[cols].clip(lower=1)
submission.to_csv('LGBM_FIN_777.csv', index=False, encoding='utf-8-sig')
submission.head(10)

In [ ]:
import os
import random
import glob
import re

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

import itertools
from datetime import timedelta
from tqdm import tqdm

seed = 427
LOOKBACK, PREDICT, BATCH_SIZE, EPOCHS = 28, 7, 16, 50
DROP_COUNT = 17

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(seed)

train = pd.read_csv('train/train.csv')
holiday = pd.to_datetime([
    '2023-01-01', '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24', '2023-03-01', '2023-05-05', '2023-05-27', '2023-05-29', '2023-06-06', '2023-08-15', '2023-09-28', '2023-09-29', '2023-09-30', '2023-10-01', '2023-10-02', '2023-10-03', '2023-10-09', '2023-12-25',
    '2024-01-01', '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12', '2024-03-01', '2024-04-10', '2024-05-06', '2024-05-15', '2024-06-06', '2024-08-15', '2024-09-16', '2024-09-17', '2024-09-18', '2024-10-01', '2024-10-03', '2024-10-09', '2024-12-25',
    '2025-01-01', '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30', '2025-03-03', '2025-05-05', '2025-05-06', '2025-06-03', '2025-06-06', '2025-08-15',
])


def check_holiday(date: pd.Timestamp) -> bool:
    return date.dayofweek in (5, 6) or date in holiday


def filter_long_zero_runs(X, y):
    lag_cols = [f'lag_{i}' for i in range(1, LOOKBACK+1)]
    def _max_zero_run(arr):
        return max((len(list(g)) for v, g in itertools.groupby(arr) if v == 0.0), default=0)
    X['max_zero_run'] = X[lag_cols].apply(_max_zero_run, axis=1)
    mask = X['max_zero_run'] < DROP_COUNT
    return X.loc[mask].drop(columns='max_zero_run'), y.loc[mask]


def interpolate_single_zero_lags(X):
    lag_cols = [f'lag_{i}' for i in range(1, LOOKBACK+1)]
    arr = X[lag_cols].astype(float).to_numpy(copy=True)
    L = arr.shape[1]
    mask = np.zeros_like(arr, dtype=bool)

    for j in range(1, L-1):
        mask[:, j] = (arr[:, j]==0) & (arr[:, j-1]!=0) & (arr[:, j+1]!=0)

    arr[mask] = np.nan
    df_interp = pd.DataFrame(arr, columns=lag_cols, index=X.index).interpolate(axis=1, method='linear').fillna(0)
    X[lag_cols] = df_interp
    return X


def preprocess(df, is_train=True):
    df.loc[df['매출수량'] < 0, '매출수량'] = 0
    df['영업일자'] = pd.to_datetime(df['영업일자'])
    df['dow'] = df['영업일자'].dt.dayofweek
    df['month'] = df['영업일자'].dt.month
    df['holiday'] = df['영업일자'].map(check_holiday).astype(int)

    def make_feature_row(key, ref_date, lag_block):
        data = {
            '영업장명_메뉴명': key,
            'dow': ref_date.dayofweek,
            'month': ref_date.month,
            'holiday': int(check_holiday(ref_date)),
            'sin_day': np.sin(2*np.pi*ref_date.timetuple().tm_yday/365.0),
            'cos_day': np.cos(2*np.pi*ref_date.timetuple().tm_yday/365.0),
            'ref_date': ref_date,
        }
        data.update({f'holiday_plus_{d}d': int(check_holiday(ref_date + pd.Timedelta(days=d))) for d in range(1, 8)})
        data.update({f'lag_{j+1}': float(val) for j, val in enumerate(lag_block)})
        return data


    X_data, y_data = [], []
    for key, group in df.groupby('영업장명_메뉴명'):
        group = group.sort_values('영업일자')
        sales, dates = group['매출수량'].to_numpy(), group['영업일자']

        if len(sales) < LOOKBACK:
            continue

        if is_train:
            Xy = [
                (make_feature_row(key, dates.iloc[i], sales[i-LOOKBACK:i][::-1]), sales[i:i+PREDICT])
                for i in range(LOOKBACK, len(sales) - PREDICT + 1)
            ]
            X_data.extend([r for r, _ in Xy])
            y_data.extend([y for _, y in Xy])
        else:
            ref_date = dates.iloc[-1] + timedelta(days=1)
            lag_block = sales[-LOOKBACK:][::-1]
            X_data.append(make_feature_row(key, ref_date, lag_block))

    X = pd.DataFrame.from_records(X_data)
    y = pd.DataFrame(y_data, columns=[f'target_{i}' for i in range(1, PREDICT+1)]) if is_train else None

    if is_train:
        X, y = filter_long_zero_runs(X, y)

    X = interpolate_single_zero_lags(X)

    # special_stores = {'느티나무 BBQ','라그로타','화담숲주막','화담숲카페'}
    # X['holiday2'] = ((X['영업장명_메뉴명'].str.split('_').str[0].isin(special_stores)) & (X['dow']==0)).astype(int)

    return X, y

def round_and_clip_min1(arr):
    a = np.rint(np.asarray(arr, dtype=float))
    a = np.where(a < 1.0, 1.0, a)
    return a

x_train, y_train = preprocess(train, is_train=True)

from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

onehot = OneHotEncoder(handle_unknown='ignore')

categorical_features = ['영업장명_메뉴명', 'dow', 'month']
numerical_features = ['holiday'] + x_train.filter(regex='holiday_plus').columns.tolist() + ['sin_day', 'cos_day'] + [f'lag_{l}' for l in range(1, LOOKBACK + 1)]

x_cat_train = onehot.fit_transform(x_train[categorical_features]).toarray()
x_num_train = x_train[numerical_features].values
x_train_processed = np.hstack([x_cat_train, x_num_train])

model_params = {
    'n_estimators': 1000,
    'subsample': 0.8,
    'max_depth': 6,
    'colsample_bytree': 0.8,
    'learning_rate': 0.05,
    'objective': 'reg:squarederror',
    'random_state': seed,
}
model = XGBRegressor(**model_params)
multi_output_regressor = MultiOutputRegressor(model)

multi_output_regressor.fit(x_train_processed, y_train)

all_preds = []

test_files = sorted(glob.glob('test/TEST_*.csv'))
for path in test_files:
    results = []

    test_df = pd.read_csv(path)
    x_test, _ = preprocess(test_df, is_train=False)

    x_cat_test = onehot.transform(x_test[categorical_features]).toarray()
    x_num_test = x_test[numerical_features].values
    x_test_processed = np.hstack([x_cat_test, x_num_test])

    preds = multi_output_regressor.predict(x_test_processed)

    for i in range(len(x_test)):
        menu = x_test.iloc[i]['영업장명_메뉴명']

        for j in range(PREDICT):
            date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
            results.append({
                '영업일자': date,
                '영업장명_메뉴명': menu,
                '매출수량': preds[i][j]
            })

    pred_df = pd.DataFrame(results)
    all_preds.append(pred_df)

full_pred_df = pd.concat(all_preds, ignore_index=True)

from collections import defaultdict

# 기존 전체 모델 예측 결과 저장
all_preds_global = full_pred_df.copy()

# 개별 모델 예측 결과 저장
all_preds_local = []

unique_keys = train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).unique()

for key in tqdm(unique_keys):
    x_train_key, y_train_key = x_train[x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key], y_train[x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key]

    # 카테고리 인코딩 (key만 존재하므로 카테고리 의미 없음)
    x_cat_train_key = onehot.fit_transform(x_train_key[categorical_features]).toarray()
    x_num_train_key = x_train_key[numerical_features].values
    x_train_processed_key = np.hstack([x_cat_train_key, x_num_train_key])

    # 개별 모델 학습
    model_key = XGBRegressor(**model_params)
    multi_reg_key = MultiOutputRegressor(model_key)
    multi_reg_key.fit(x_train_processed_key, y_train_key)

    # 각 test 파일별 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)

        x_test_key = x_test[x_test['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key]

        x_cat_test_key = onehot.transform(x_test_key[categorical_features]).toarray()
        x_num_test_key = x_test_key[numerical_features].values
        x_test_processed_key = np.hstack([x_cat_test_key, x_num_test_key])

        preds_key = multi_reg_key.predict(x_test_processed_key)

        results_key = []
        for i in range(len(x_test_key)):
            menu = x_test_key.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_key.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_key[i][j]
                })

        pred_df_key = pd.DataFrame(results_key)
        all_preds_local.append(pred_df_key)

# 개별 모델 전체 결과 합치기
full_pred_df_local = pd.concat(all_preds_local, ignore_index=True)
all_preds_local_menu = []
unique_store_menu = train['영업장명_메뉴명'].unique()

for sm in tqdm(unique_store_menu):
    y_train_sm = y_train[x_train['영업장명_메뉴명'] == sm]
    x_train_sm = x_train[x_train['영업장명_메뉴명'] == sm]

    # 카테고리 / 수치 분리
    x_cat_sm = onehot.fit_transform(x_train_sm[categorical_features]).toarray()
    x_num_sm = x_train_sm[numerical_features].values
    x_train_processed_sm = np.hstack([x_cat_sm, x_num_sm])

    model_sm = XGBRegressor(**model_params)
    multi_reg_sm = MultiOutputRegressor(model_sm)
    multi_reg_sm.fit(x_train_processed_sm, y_train_sm)

    # 테스트 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)
        x_test_sm = x_test[x_test['영업장명_메뉴명'] == sm]

        if len(x_test_sm) == 0:
            continue  # 해당 메뉴 없는 경우 건너뛰기

        x_cat_test_sm = onehot.transform(x_test_sm[categorical_features]).toarray()
        x_num_test_sm = x_test_sm[numerical_features].values
        x_test_processed_sm = np.hstack([x_cat_test_sm, x_num_test_sm])

        preds_sm = multi_reg_sm.predict(x_test_processed_sm)

        results_sm = []
        for i in range(len(x_test_sm)):
            menu = x_test_sm.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_sm.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_sm[i][j]
                })
        all_preds_local_menu.append(pd.DataFrame(results_sm))

full_pred_df_menu = pd.concat(all_preds_local_menu, ignore_index=True)

# group = {
#     0: ['담하', '미라시아', '느티나무 셀프BBQ', '포레스트릿', '카페테리아'],
#     1: ['라그로타'],
#     2: ['연회장'],
#     3: ['화담숲주막', '화담숲카페']
# }

group = {
    0: ['담하', '미라시아', '느티나무 셀프BBQ', '포레스트릿'],
    1: ['라그로타', '카페테리아'],
    2: ['연회장'],
    3: ['화담숲주막', '화담숲카페']
}

all_preds_loc = []

for group_id, store_list in tqdm(group.items()):
    # 해당 그룹에 속하는 train 데이터 선택
    mask_train = x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).isin(store_list)
    x_train_group = x_train[mask_train]
    y_train_group = y_train[mask_train]

    # 카테고리 인코딩
    x_cat_train_group = onehot.fit_transform(x_train_group[categorical_features]).toarray()
    x_num_train_group = x_train_group[numerical_features].values
    x_train_processed_group = np.hstack([x_cat_train_group, x_num_train_group])

    # 그룹별 모델 학습
    model_group = XGBRegressor(**model_params)
    multi_reg_group = MultiOutputRegressor(model_group)
    multi_reg_group.fit(x_train_processed_group, y_train_group)

    # 각 test 파일별 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)

        # 해당 그룹 test 데이터 선택
        mask_test = x_test['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).isin(store_list)
        x_test_group = x_test[mask_test]

        if len(x_test_group) == 0:
            continue  # 해당 그룹 데이터가 없으면 스킵

        x_cat_test_group = onehot.transform(x_test_group[categorical_features]).toarray()
        x_num_test_group = x_test_group[numerical_features].values
        x_test_processed_group = np.hstack([x_cat_test_group, x_num_test_group])

        preds_group = multi_reg_group.predict(x_test_processed_group)

        results_group = []
        for i in range(len(x_test_group)):
            menu = x_test_group.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_group.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_group[i][j]
                })

        pred_df_group = pd.DataFrame(results_group)
        all_preds_loc.append(pred_df_group)

# 그룹별 모델 전체 결과 합치기
full_pred_df_loc = pd.concat(all_preds_loc, ignore_index=True)

menu_groups = {
    "느티나무 셀프BBQ": {
        "메인메뉴": [
            "느티나무 셀프BBQ_BBQ55(단체)",
            "느티나무 셀프BBQ_본삼겹 (단품,실내)",
            "느티나무 셀프BBQ_신라면",
            "느티나무 셀프BBQ_육개장 사발면",
            "느티나무 셀프BBQ_햇반"
        ],
        "사이드": [
            "느티나무 셀프BBQ_쌈야채세트",
            "느티나무 셀프BBQ_쌈장",
            "느티나무 셀프BBQ_허브솔트"
        ],
        "음료": [
            "느티나무 셀프BBQ_참이슬 (단체)",
            "느티나무 셀프BBQ_카스 병(단체)",
            "느티나무 셀프BBQ_콜라 (단체)",
            "느티나무 셀프BBQ_스프라이트 (단체)"
        ],
        "소모품": [
            "느티나무 셀프BBQ_1인 수저세트",
            "느티나무 셀프BBQ_일회용 소주컵",
            "느티나무 셀프BBQ_일회용 종이컵",
            "느티나무 셀프BBQ_친환경 접시 14cm",
            "느티나무 셀프BBQ_친환경 접시 23cm"
        ],
        "대여": [
            "느티나무 셀프BBQ_대여료 30,000원",
            "느티나무 셀프BBQ_대여료 60,000원",
            "느티나무 셀프BBQ_대여료 90,000원",
            "느티나무 셀프BBQ_잔디그늘집 대여료 (6인석)",
            "느티나무 셀프BBQ_잔디그늘집 대여료 (12인석)",
            "느티나무 셀프BBQ_잔디그늘집 의자 추가"
        ]
    },
    "담하": {
        "메인메뉴": [
            "담하_(단체) 생목살 김치전골 2.0",
            "담하_(단체) 은이버섯 갈비탕",
            "담하_(단체) 한우 우거지 국밥",
            "담하_(단체) 황태해장국 3/27까지",
            "담하_(정식) 된장찌개",
            "담하_(정식) 물냉면 ",
            "담하_(정식) 비빔냉면",
            "담하_(후식) 된장찌개",
            "담하_(후식) 물냉면",
            "담하_(후식) 비빔냉면",
            "담하_갑오징어 비빔밥",
            "담하_갱시기",
            "담하_꼬막 비빔밥",
            "담하_담하 한우 불고기",
            "담하_담하 한우 불고기 정식",
            "담하_더덕 한우 지짐",
            "담하_들깨 양지탕",
            "담하_명태회 비빔냉면",
            "담하_봉평메밀 물냉면",
            "담하_생목살 김치찌개",
            "담하_은이버섯 갈비탕",
            "담하_한우 떡갈비 정식",
            "담하_한우 미역국 정식",
            "담하_한우 우거지 국밥",
            "담하_한우 차돌박이 된장찌개",
            "담하_황태해장국"
        ],
        "사이드": [
            "담하_(단체) 공깃밥",
            "담하_공깃밥",
            "담하_라면사리",
            "담하_메밀면 사리"
        ],
        "음료": [
            "담하_느린마을 막걸리",
            "담하_명인안동소주",
            "담하_문막 복분자 칵테일",
            "담하_스프라이트",
            "담하_제로콜라",
            "담하_참이슬",
            "담하_처음처럼",
            "담하_카스",
            "담하_콜라",
            "담하_테라",
            "담하_하동 매실 칵테일"
        ],
        "대여": [
            "담하_룸 이용료"
        ]
    },
    "라그로타": {
        "메인메뉴": [
            "라그로타_AUS (200g)",
            "라그로타_한우 (200g)",
            "라그로타_양갈비 (4ps)",
            "라그로타_까르보나라",
            "라그로타_알리오 에 올리오 ",
            "라그로타_버섯 크림 리조또",
            "라그로타_해산물 토마토 리조또",
            "라그로타_해산물 토마토 스파게티",
            "라그로타_해산물 토마토 스튜 파스타",
            "라그로타_모둠 해산물 플래터"
        ],
        "샐러드·사이드": [
            "라그로타_그릴드 비프 샐러드",
            "라그로타_시저 샐러드 ",
            "라그로타_빵 추가 (1인)",
            "라그로타_Open Food"
        ],
        "음료": [
            "라그로타_아메리카노",
            "라그로타_자몽리치에이드",
            "라그로타_스프라이트",
            "라그로타_제로콜라",
            "라그로타_콜라",
            "라그로타_G-Charge(3)",
            "라그로타_Gls.Sileni",
            "라그로타_Gls.미션 서드",
            "라그로타_미션 서드 카베르네 쉬라",
            "라그로타_카스",
            "라그로타_하이네켄(생)"
        ],
    },
    "미라시아": {
        "브런치·패키지": [
            "미라시아_(단체)브런치주중 36,000",
            "미라시아_미라시아 브런치 (패키지)",
            "미라시아_브런치 2인 패키지 ",
            "미라시아_브런치 4인 패키지 ",
            "미라시아_브런치(대인) 주말",
            "미라시아_브런치(대인) 주중",
            "미라시아_브런치(어린이)"
        ],
        "플래터·피자·파스타": [
            "미라시아_(오븐) 하와이안 쉬림프 피자",
            "미라시아_(화덕) 불고기 페퍼로니 반반피자",
            "미라시아_BBQ Platter",
            "미라시아_BBQ 고기추가",
            "미라시아_보일링 랍스타 플래터",
            "미라시아_보일링 랍스타 플래터(덜매운맛)",
            "미라시아_쉬림프 투움바 파스타",
            "미라시아_오븐구이 윙과 킬바사소세지"
        ],
        "사이드·추가": [
            "미라시아_공깃밥",
            "미라시아_칠리 치즈 프라이",
            "미라시아_파스타면 추가(150g)",
            "미라시아_콥 샐러드"
        ],
        "음료": [
            "미라시아_스프라이트",
            "미라시아_애플망고 에이드",
            "미라시아_핑크레몬에이드",
            "미라시아_코카콜라",
            "미라시아_코카콜라(제로)",
            "미라시아_글라스와인 (레드)",
            "미라시아_레인보우칵테일(알코올)",
            "미라시아_버드와이저(무제한)",
            "미라시아_스텔라(무제한)",
            "미라시아_얼그레이 하이볼",
            "미라시아_유자 하이볼",
            "미라시아_잭 애플 토닉"
        ],
    },
    "연회장": {
        "공간대여": [
            "연회장_Conference L1",
            "연회장_Conference L2",
            "연회장_Conference L3",
            "연회장_Conference M1",
            "연회장_Conference M8",
            "연회장_Conference M9",
            "연회장_Convention Hall",
            "연회장_Grand Ballroom",
            "연회장_OPUS 2"
        ],
        "메인요리": [
            "연회장_돈목살 김치찌개 (밥포함)",
            "연회장_마라샹궈",
            "연회장_매콤 무뼈닭발&계란찜",
            "연회장_모둠 돈육구이(3인)",
            "연회장_왕갈비치킨"
        ],
        "기타": [
            "연회장_공깃밥",
            "연회장_삼겹살추가 (200g)",
            "연회장_야채추가",
            "연회장_주먹밥 (2ea)",
            "연회장_Cass Beer",
            "연회장_Regular Coffee",
            "연회장_Cookie Platter",
            "연회장_골뱅이무침",
            "연회장_로제 치즈떡볶이"
        ],
    },
    "카페테리아": {
        "정식·단체식": [
            "카페테리아_단체식 13000(신)",
            "카페테리아_단체식 18000(신)",
            "카페테리아_한상 삼겹구이 정식(2인) 소요시간 약 15~20분",
            "카페테리아_오픈푸드"
        ],
        "단품메뉴": [
            "카페테리아_돼지고기 김치찌개",
            "카페테리아_수제 등심 돈까스",
            "카페테리아_치즈돈까스",
            "카페테리아_어린이 돈까스",
            "카페테리아_약 고추장 돌솥비빔밥",
            "카페테리아_진사골 설렁탕"
        ],
        "면·밥류": [
            "카페테리아_새우 볶음밥",
            "카페테리아_새우튀김 우동",
            "카페테리아_짜장면",
            "카페테리아_짜장밥",
            "카페테리아_짬뽕",
            "카페테리아_짬뽕밥"
        ],
        "음료": [
            "카페테리아_아메리카노(HOT)",
            "카페테리아_아메리카노(ICE)",
            "카페테리아_카페라떼(HOT)",
            "카페테리아_카페라떼(ICE)",
            "카페테리아_복숭아 아이스티"
        ],
        "사이드·추가": [
            "카페테리아_공깃밥(추가)",
            "카페테리아_샷 추가",
            "카페테리아_구슬아이스크림"
        ]
    },
    "포레스트릿": {
        "분식": [
            "포레스트릿_떡볶이",
            "포레스트릿_꼬치어묵",
            "포레스트릿_치즈 핫도그",
            "포레스트릿_페스츄리 소시지"
        ],
        "음료": [
            "포레스트릿_아메리카노(HOT)",
            "포레스트릿_아메리카노(ICE)",
            "포레스트릿_카페라떼(HOT)",
            "포레스트릿_카페라떼(ICE)",
            "포레스트릿_복숭아 아이스티",
            "포레스트릿_코카콜라",
            "포레스트릿_스프라이트",
            "포레스트릿_생수"
        ]
    },
    "화담숲주막": {
        "음료": [
            "화담숲주막_느린마을 막걸리",
            "화담숲주막_참살이 막걸리",
            "화담숲주막_단호박 식혜 ",
            "화담숲주막_찹쌀식혜",
            "화담숲주막_콜라",
            "화담숲주막_스프라이트"
        ],
        "안주": [
            "화담숲주막_해물파전",
            "화담숲주막_병천순대"
        ]
    },
    "화담숲카페": {
        "전통 음료": [
            "화담숲카페_메밀미숫가루",
            "화담숲카페_현미뻥스크림"
        ],
        "커피": [
            "화담숲카페_아메리카노 HOT",
            "화담숲카페_아메리카노 ICE",
            "화담숲카페_카페라떼 ICE"
        ]
    }

}

all_preds_group = []

for store_name, groups in tqdm(menu_groups.items()):
    for group_name, menu_list in groups.items():
        # 그룹 데이터 추출
        x_train_group = x_train[x_train['영업장명_메뉴명'].isin(menu_list)]
        y_train_group = y_train.loc[x_train_group.index]

        if len(x_train_group) == 0:
            continue

        # 카테고리 인코딩
        x_cat_train_group = onehot.fit_transform(x_train_group[categorical_features]).toarray()
        x_num_train_group = x_train_group[numerical_features].values
        x_train_processed_group = np.hstack([x_cat_train_group, x_num_train_group])

        # 그룹별 모델 학습
        model_group = XGBRegressor(**model_params)
        multi_reg_group = MultiOutputRegressor(model_group)
        multi_reg_group.fit(x_train_processed_group, y_train_group)

        # 각 test 파일별 예측
        for path in test_files:
            test_df = pd.read_csv(path)
            x_test, _ = preprocess(test_df, is_train=False)

            x_test_group = x_test[x_test['영업장명_메뉴명'].isin(menu_list)]
            if len(x_test_group) == 0:
                continue

            x_cat_test_group = onehot.transform(x_test_group[categorical_features]).toarray()
            x_num_test_group = x_test_group[numerical_features].values
            x_test_processed_group = np.hstack([x_cat_test_group, x_num_test_group])

            preds_group = multi_reg_group.predict(x_test_processed_group)

            results_group = []
            for i in range(len(x_test_group)):
                menu = x_test_group.iloc[i]['영업장명_메뉴명']
                for j in range(PREDICT):
                    date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                    results_group.append({
                        '영업일자': date,
                        '영업장명_메뉴명': menu,
                        '매출수량': preds_group[i][j],
                        '그룹명': group_name   # 그룹 단위 정보 추가
                    })

            pred_df_group = pd.DataFrame(results_group)
            all_preds_group.append(pred_df_group)

full_pred_df_group = pd.concat(all_preds_group, ignore_index=True)

# === 앙상블 ===
ensemble_preds = all_preds_global.copy()

full_pred_df_local = full_pred_df_local.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_local = full_pred_df_local.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_menu = full_pred_df_menu.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_menu = full_pred_df_menu.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_loc = full_pred_df_loc.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_loc = full_pred_df_loc.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_group = full_pred_df_group.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_group = full_pred_df_group.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

ensemble_preds['매출수량'] = ensemble_preds['매출수량']*(1/5) + full_pred_df_local['매출수량']*(1/5) + full_pred_df_menu['매출수량']*(1/5) + full_pred_df_loc['매출수량']*(1/5) + full_pred_df_group['매출수량']*(1/5)

# 제출 파일 생성
submission = pd.read_csv('sample_submission.csv')
cols = submission.columns.drop('영업일자')
for col in cols:
    df = ensemble_preds[ensemble_preds['영업장명_메뉴명'] == col]
    df = df.set_index('영업일자')
    df = df.loc[submission['영업일자']].reset_index()
    submission[col] = round_and_clip_min1(df['매출수량'])

submission[cols] = submission[cols].clip(lower=1)
submission.to_csv('LGBM_FIN_427.csv', index=False, encoding='utf-8-sig')
submission.head(10)

In [ ]:
import os
import random
import glob
import re

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

import itertools
from datetime import timedelta
from tqdm import tqdm

seed = 21011928
LOOKBACK, PREDICT, BATCH_SIZE, EPOCHS = 28, 7, 16, 50
DROP_COUNT = 17

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(seed)

train = pd.read_csv('train/train.csv')
holiday = pd.to_datetime([
    '2023-01-01', '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24', '2023-03-01', '2023-05-05', '2023-05-27', '2023-05-29', '2023-06-06', '2023-08-15', '2023-09-28', '2023-09-29', '2023-09-30', '2023-10-01', '2023-10-02', '2023-10-03', '2023-10-09', '2023-12-25',
    '2024-01-01', '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12', '2024-03-01', '2024-04-10', '2024-05-06', '2024-05-15', '2024-06-06', '2024-08-15', '2024-09-16', '2024-09-17', '2024-09-18', '2024-10-01', '2024-10-03', '2024-10-09', '2024-12-25',
    '2025-01-01', '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30', '2025-03-03', '2025-05-05', '2025-05-06', '2025-06-03', '2025-06-06', '2025-08-15',
])


def check_holiday(date: pd.Timestamp) -> bool:
    return date.dayofweek in (5, 6) or date in holiday


def filter_long_zero_runs(X, y):
    lag_cols = [f'lag_{i}' for i in range(1, LOOKBACK+1)]
    def _max_zero_run(arr):
        return max((len(list(g)) for v, g in itertools.groupby(arr) if v == 0.0), default=0)
    X['max_zero_run'] = X[lag_cols].apply(_max_zero_run, axis=1)
    mask = X['max_zero_run'] < DROP_COUNT
    return X.loc[mask].drop(columns='max_zero_run'), y.loc[mask]


def interpolate_single_zero_lags(X):
    lag_cols = [f'lag_{i}' for i in range(1, LOOKBACK+1)]
    arr = X[lag_cols].astype(float).to_numpy(copy=True)
    L = arr.shape[1]
    mask = np.zeros_like(arr, dtype=bool)

    for j in range(1, L-1):
        mask[:, j] = (arr[:, j]==0) & (arr[:, j-1]!=0) & (arr[:, j+1]!=0)

    arr[mask] = np.nan
    df_interp = pd.DataFrame(arr, columns=lag_cols, index=X.index).interpolate(axis=1, method='linear').fillna(0)
    X[lag_cols] = df_interp
    return X


def preprocess(df, is_train=True):
    df.loc[df['매출수량'] < 0, '매출수량'] = 0
    df['영업일자'] = pd.to_datetime(df['영업일자'])
    df['dow'] = df['영업일자'].dt.dayofweek
    df['month'] = df['영업일자'].dt.month
    df['holiday'] = df['영업일자'].map(check_holiday).astype(int)

    def make_feature_row(key, ref_date, lag_block):
        data = {
            '영업장명_메뉴명': key,
            'dow': ref_date.dayofweek,
            'month': ref_date.month,
            'holiday': int(check_holiday(ref_date)),
            'sin_day': np.sin(2*np.pi*ref_date.timetuple().tm_yday/365.0),
            'cos_day': np.cos(2*np.pi*ref_date.timetuple().tm_yday/365.0),
            'ref_date': ref_date,
        }
        data.update({f'holiday_plus_{d}d': int(check_holiday(ref_date + pd.Timedelta(days=d))) for d in range(1, 8)})
        data.update({f'lag_{j+1}': float(val) for j, val in enumerate(lag_block)})
        return data


    X_data, y_data = [], []
    for key, group in df.groupby('영업장명_메뉴명'):
        group = group.sort_values('영업일자')
        sales, dates = group['매출수량'].to_numpy(), group['영업일자']

        if len(sales) < LOOKBACK:
            continue

        if is_train:
            Xy = [
                (make_feature_row(key, dates.iloc[i], sales[i-LOOKBACK:i][::-1]), sales[i:i+PREDICT])
                for i in range(LOOKBACK, len(sales) - PREDICT + 1)
            ]
            X_data.extend([r for r, _ in Xy])
            y_data.extend([y for _, y in Xy])
        else:
            ref_date = dates.iloc[-1] + timedelta(days=1)
            lag_block = sales[-LOOKBACK:][::-1]
            X_data.append(make_feature_row(key, ref_date, lag_block))

    X = pd.DataFrame.from_records(X_data)
    y = pd.DataFrame(y_data, columns=[f'target_{i}' for i in range(1, PREDICT+1)]) if is_train else None

    if is_train:
        X, y = filter_long_zero_runs(X, y)

    X = interpolate_single_zero_lags(X)

    # special_stores = {'느티나무 BBQ','라그로타','화담숲주막','화담숲카페'}
    # X['holiday2'] = ((X['영업장명_메뉴명'].str.split('_').str[0].isin(special_stores)) & (X['dow']==0)).astype(int)

    return X, y

def round_and_clip_min1(arr):
    a = np.rint(np.asarray(arr, dtype=float))
    a = np.where(a < 1.0, 1.0, a)
    return a

x_train, y_train = preprocess(train, is_train=True)

from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

onehot = OneHotEncoder(handle_unknown='ignore')

categorical_features = ['영업장명_메뉴명', 'dow', 'month']
numerical_features = ['holiday'] + x_train.filter(regex='holiday_plus').columns.tolist() + ['sin_day', 'cos_day'] + [f'lag_{l}' for l in range(1, LOOKBACK + 1)]

x_cat_train = onehot.fit_transform(x_train[categorical_features]).toarray()
x_num_train = x_train[numerical_features].values
x_train_processed = np.hstack([x_cat_train, x_num_train])

model_params = {
    'n_estimators': 1000,
    'subsample': 0.8,
    'max_depth': 6,
    'colsample_bytree': 0.8,
    'learning_rate': 0.05,
    'objective': 'reg:squarederror',
    'random_state': seed,
}
model = XGBRegressor(**model_params)
multi_output_regressor = MultiOutputRegressor(model)

multi_output_regressor.fit(x_train_processed, y_train)

all_preds = []

test_files = sorted(glob.glob('test/TEST_*.csv'))
for path in test_files:
    results = []

    test_df = pd.read_csv(path)
    x_test, _ = preprocess(test_df, is_train=False)

    x_cat_test = onehot.transform(x_test[categorical_features]).toarray()
    x_num_test = x_test[numerical_features].values
    x_test_processed = np.hstack([x_cat_test, x_num_test])

    preds = multi_output_regressor.predict(x_test_processed)

    for i in range(len(x_test)):
        menu = x_test.iloc[i]['영업장명_메뉴명']

        for j in range(PREDICT):
            date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
            results.append({
                '영업일자': date,
                '영업장명_메뉴명': menu,
                '매출수량': preds[i][j]
            })

    pred_df = pd.DataFrame(results)
    all_preds.append(pred_df)

full_pred_df = pd.concat(all_preds, ignore_index=True)

from collections import defaultdict

# 기존 전체 모델 예측 결과 저장
all_preds_global = full_pred_df.copy()

# 개별 모델 예측 결과 저장
all_preds_local = []

unique_keys = train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).unique()

for key in tqdm(unique_keys):
    x_train_key, y_train_key = x_train[x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key], y_train[x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key]

    # 카테고리 인코딩 (key만 존재하므로 카테고리 의미 없음)
    x_cat_train_key = onehot.fit_transform(x_train_key[categorical_features]).toarray()
    x_num_train_key = x_train_key[numerical_features].values
    x_train_processed_key = np.hstack([x_cat_train_key, x_num_train_key])

    # 개별 모델 학습
    model_key = XGBRegressor(**model_params)
    multi_reg_key = MultiOutputRegressor(model_key)
    multi_reg_key.fit(x_train_processed_key, y_train_key)

    # 각 test 파일별 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)

        x_test_key = x_test[x_test['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key]

        x_cat_test_key = onehot.transform(x_test_key[categorical_features]).toarray()
        x_num_test_key = x_test_key[numerical_features].values
        x_test_processed_key = np.hstack([x_cat_test_key, x_num_test_key])

        preds_key = multi_reg_key.predict(x_test_processed_key)

        results_key = []
        for i in range(len(x_test_key)):
            menu = x_test_key.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_key.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_key[i][j]
                })

        pred_df_key = pd.DataFrame(results_key)
        all_preds_local.append(pred_df_key)

# 개별 모델 전체 결과 합치기
full_pred_df_local = pd.concat(all_preds_local, ignore_index=True)
all_preds_local_menu = []
unique_store_menu = train['영업장명_메뉴명'].unique()

for sm in tqdm(unique_store_menu):
    y_train_sm = y_train[x_train['영업장명_메뉴명'] == sm]
    x_train_sm = x_train[x_train['영업장명_메뉴명'] == sm]

    # 카테고리 / 수치 분리
    x_cat_sm = onehot.fit_transform(x_train_sm[categorical_features]).toarray()
    x_num_sm = x_train_sm[numerical_features].values
    x_train_processed_sm = np.hstack([x_cat_sm, x_num_sm])

    model_sm = XGBRegressor(**model_params)
    multi_reg_sm = MultiOutputRegressor(model_sm)
    multi_reg_sm.fit(x_train_processed_sm, y_train_sm)

    # 테스트 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)
        x_test_sm = x_test[x_test['영업장명_메뉴명'] == sm]

        if len(x_test_sm) == 0:
            continue  # 해당 메뉴 없는 경우 건너뛰기

        x_cat_test_sm = onehot.transform(x_test_sm[categorical_features]).toarray()
        x_num_test_sm = x_test_sm[numerical_features].values
        x_test_processed_sm = np.hstack([x_cat_test_sm, x_num_test_sm])

        preds_sm = multi_reg_sm.predict(x_test_processed_sm)

        results_sm = []
        for i in range(len(x_test_sm)):
            menu = x_test_sm.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_sm.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_sm[i][j]
                })
        all_preds_local_menu.append(pd.DataFrame(results_sm))

full_pred_df_menu = pd.concat(all_preds_local_menu, ignore_index=True)

# group = {
#     0: ['담하', '미라시아', '느티나무 셀프BBQ', '포레스트릿', '카페테리아'],
#     1: ['라그로타'],
#     2: ['연회장'],
#     3: ['화담숲주막', '화담숲카페']
# }

group = {
    0: ['담하', '미라시아', '느티나무 셀프BBQ', '포레스트릿'],
    1: ['라그로타', '카페테리아'],
    2: ['연회장'],
    3: ['화담숲주막', '화담숲카페']
}

all_preds_loc = []

for group_id, store_list in tqdm(group.items()):
    # 해당 그룹에 속하는 train 데이터 선택
    mask_train = x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).isin(store_list)
    x_train_group = x_train[mask_train]
    y_train_group = y_train[mask_train]

    # 카테고리 인코딩
    x_cat_train_group = onehot.fit_transform(x_train_group[categorical_features]).toarray()
    x_num_train_group = x_train_group[numerical_features].values
    x_train_processed_group = np.hstack([x_cat_train_group, x_num_train_group])

    # 그룹별 모델 학습
    model_group = XGBRegressor(**model_params)
    multi_reg_group = MultiOutputRegressor(model_group)
    multi_reg_group.fit(x_train_processed_group, y_train_group)

    # 각 test 파일별 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)

        # 해당 그룹 test 데이터 선택
        mask_test = x_test['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).isin(store_list)
        x_test_group = x_test[mask_test]

        if len(x_test_group) == 0:
            continue  # 해당 그룹 데이터가 없으면 스킵

        x_cat_test_group = onehot.transform(x_test_group[categorical_features]).toarray()
        x_num_test_group = x_test_group[numerical_features].values
        x_test_processed_group = np.hstack([x_cat_test_group, x_num_test_group])

        preds_group = multi_reg_group.predict(x_test_processed_group)

        results_group = []
        for i in range(len(x_test_group)):
            menu = x_test_group.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_group.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_group[i][j]
                })

        pred_df_group = pd.DataFrame(results_group)
        all_preds_loc.append(pred_df_group)

# 그룹별 모델 전체 결과 합치기
full_pred_df_loc = pd.concat(all_preds_loc, ignore_index=True)

menu_groups = {
    "느티나무 셀프BBQ": {
        "메인메뉴": [
            "느티나무 셀프BBQ_BBQ55(단체)",
            "느티나무 셀프BBQ_본삼겹 (단품,실내)",
            "느티나무 셀프BBQ_신라면",
            "느티나무 셀프BBQ_육개장 사발면",
            "느티나무 셀프BBQ_햇반"
        ],
        "사이드": [
            "느티나무 셀프BBQ_쌈야채세트",
            "느티나무 셀프BBQ_쌈장",
            "느티나무 셀프BBQ_허브솔트"
        ],
        "음료": [
            "느티나무 셀프BBQ_참이슬 (단체)",
            "느티나무 셀프BBQ_카스 병(단체)",
            "느티나무 셀프BBQ_콜라 (단체)",
            "느티나무 셀프BBQ_스프라이트 (단체)"
        ],
        "소모품": [
            "느티나무 셀프BBQ_1인 수저세트",
            "느티나무 셀프BBQ_일회용 소주컵",
            "느티나무 셀프BBQ_일회용 종이컵",
            "느티나무 셀프BBQ_친환경 접시 14cm",
            "느티나무 셀프BBQ_친환경 접시 23cm"
        ],
        "대여": [
            "느티나무 셀프BBQ_대여료 30,000원",
            "느티나무 셀프BBQ_대여료 60,000원",
            "느티나무 셀프BBQ_대여료 90,000원",
            "느티나무 셀프BBQ_잔디그늘집 대여료 (6인석)",
            "느티나무 셀프BBQ_잔디그늘집 대여료 (12인석)",
            "느티나무 셀프BBQ_잔디그늘집 의자 추가"
        ]
    },
    "담하": {
        "메인메뉴": [
            "담하_(단체) 생목살 김치전골 2.0",
            "담하_(단체) 은이버섯 갈비탕",
            "담하_(단체) 한우 우거지 국밥",
            "담하_(단체) 황태해장국 3/27까지",
            "담하_(정식) 된장찌개",
            "담하_(정식) 물냉면 ",
            "담하_(정식) 비빔냉면",
            "담하_(후식) 된장찌개",
            "담하_(후식) 물냉면",
            "담하_(후식) 비빔냉면",
            "담하_갑오징어 비빔밥",
            "담하_갱시기",
            "담하_꼬막 비빔밥",
            "담하_담하 한우 불고기",
            "담하_담하 한우 불고기 정식",
            "담하_더덕 한우 지짐",
            "담하_들깨 양지탕",
            "담하_명태회 비빔냉면",
            "담하_봉평메밀 물냉면",
            "담하_생목살 김치찌개",
            "담하_은이버섯 갈비탕",
            "담하_한우 떡갈비 정식",
            "담하_한우 미역국 정식",
            "담하_한우 우거지 국밥",
            "담하_한우 차돌박이 된장찌개",
            "담하_황태해장국"
        ],
        "사이드": [
            "담하_(단체) 공깃밥",
            "담하_공깃밥",
            "담하_라면사리",
            "담하_메밀면 사리"
        ],
        "음료": [
            "담하_느린마을 막걸리",
            "담하_명인안동소주",
            "담하_문막 복분자 칵테일",
            "담하_스프라이트",
            "담하_제로콜라",
            "담하_참이슬",
            "담하_처음처럼",
            "담하_카스",
            "담하_콜라",
            "담하_테라",
            "담하_하동 매실 칵테일"
        ],
        "대여": [
            "담하_룸 이용료"
        ]
    },
    "라그로타": {
        "메인메뉴": [
            "라그로타_AUS (200g)",
            "라그로타_한우 (200g)",
            "라그로타_양갈비 (4ps)",
            "라그로타_까르보나라",
            "라그로타_알리오 에 올리오 ",
            "라그로타_버섯 크림 리조또",
            "라그로타_해산물 토마토 리조또",
            "라그로타_해산물 토마토 스파게티",
            "라그로타_해산물 토마토 스튜 파스타",
            "라그로타_모둠 해산물 플래터"
        ],
        "샐러드·사이드": [
            "라그로타_그릴드 비프 샐러드",
            "라그로타_시저 샐러드 ",
            "라그로타_빵 추가 (1인)",
            "라그로타_Open Food"
        ],
        "음료": [
            "라그로타_아메리카노",
            "라그로타_자몽리치에이드",
            "라그로타_스프라이트",
            "라그로타_제로콜라",
            "라그로타_콜라",
            "라그로타_G-Charge(3)",
            "라그로타_Gls.Sileni",
            "라그로타_Gls.미션 서드",
            "라그로타_미션 서드 카베르네 쉬라",
            "라그로타_카스",
            "라그로타_하이네켄(생)"
        ],
    },
    "미라시아": {
        "브런치·패키지": [
            "미라시아_(단체)브런치주중 36,000",
            "미라시아_미라시아 브런치 (패키지)",
            "미라시아_브런치 2인 패키지 ",
            "미라시아_브런치 4인 패키지 ",
            "미라시아_브런치(대인) 주말",
            "미라시아_브런치(대인) 주중",
            "미라시아_브런치(어린이)"
        ],
        "플래터·피자·파스타": [
            "미라시아_(오븐) 하와이안 쉬림프 피자",
            "미라시아_(화덕) 불고기 페퍼로니 반반피자",
            "미라시아_BBQ Platter",
            "미라시아_BBQ 고기추가",
            "미라시아_보일링 랍스타 플래터",
            "미라시아_보일링 랍스타 플래터(덜매운맛)",
            "미라시아_쉬림프 투움바 파스타",
            "미라시아_오븐구이 윙과 킬바사소세지"
        ],
        "사이드·추가": [
            "미라시아_공깃밥",
            "미라시아_칠리 치즈 프라이",
            "미라시아_파스타면 추가(150g)",
            "미라시아_콥 샐러드"
        ],
        "음료": [
            "미라시아_스프라이트",
            "미라시아_애플망고 에이드",
            "미라시아_핑크레몬에이드",
            "미라시아_코카콜라",
            "미라시아_코카콜라(제로)",
            "미라시아_글라스와인 (레드)",
            "미라시아_레인보우칵테일(알코올)",
            "미라시아_버드와이저(무제한)",
            "미라시아_스텔라(무제한)",
            "미라시아_얼그레이 하이볼",
            "미라시아_유자 하이볼",
            "미라시아_잭 애플 토닉"
        ],
    },
    "연회장": {
        "공간대여": [
            "연회장_Conference L1",
            "연회장_Conference L2",
            "연회장_Conference L3",
            "연회장_Conference M1",
            "연회장_Conference M8",
            "연회장_Conference M9",
            "연회장_Convention Hall",
            "연회장_Grand Ballroom",
            "연회장_OPUS 2"
        ],
        "메인요리": [
            "연회장_돈목살 김치찌개 (밥포함)",
            "연회장_마라샹궈",
            "연회장_매콤 무뼈닭발&계란찜",
            "연회장_모둠 돈육구이(3인)",
            "연회장_왕갈비치킨"
        ],
        "기타": [
            "연회장_공깃밥",
            "연회장_삼겹살추가 (200g)",
            "연회장_야채추가",
            "연회장_주먹밥 (2ea)",
            "연회장_Cass Beer",
            "연회장_Regular Coffee",
            "연회장_Cookie Platter",
            "연회장_골뱅이무침",
            "연회장_로제 치즈떡볶이"
        ],
    },
    "카페테리아": {
        "정식·단체식": [
            "카페테리아_단체식 13000(신)",
            "카페테리아_단체식 18000(신)",
            "카페테리아_한상 삼겹구이 정식(2인) 소요시간 약 15~20분",
            "카페테리아_오픈푸드"
        ],
        "단품메뉴": [
            "카페테리아_돼지고기 김치찌개",
            "카페테리아_수제 등심 돈까스",
            "카페테리아_치즈돈까스",
            "카페테리아_어린이 돈까스",
            "카페테리아_약 고추장 돌솥비빔밥",
            "카페테리아_진사골 설렁탕"
        ],
        "면·밥류": [
            "카페테리아_새우 볶음밥",
            "카페테리아_새우튀김 우동",
            "카페테리아_짜장면",
            "카페테리아_짜장밥",
            "카페테리아_짬뽕",
            "카페테리아_짬뽕밥"
        ],
        "음료": [
            "카페테리아_아메리카노(HOT)",
            "카페테리아_아메리카노(ICE)",
            "카페테리아_카페라떼(HOT)",
            "카페테리아_카페라떼(ICE)",
            "카페테리아_복숭아 아이스티"
        ],
        "사이드·추가": [
            "카페테리아_공깃밥(추가)",
            "카페테리아_샷 추가",
            "카페테리아_구슬아이스크림"
        ]
    },
    "포레스트릿": {
        "분식": [
            "포레스트릿_떡볶이",
            "포레스트릿_꼬치어묵",
            "포레스트릿_치즈 핫도그",
            "포레스트릿_페스츄리 소시지"
        ],
        "음료": [
            "포레스트릿_아메리카노(HOT)",
            "포레스트릿_아메리카노(ICE)",
            "포레스트릿_카페라떼(HOT)",
            "포레스트릿_카페라떼(ICE)",
            "포레스트릿_복숭아 아이스티",
            "포레스트릿_코카콜라",
            "포레스트릿_스프라이트",
            "포레스트릿_생수"
        ]
    },
    "화담숲주막": {
        "음료": [
            "화담숲주막_느린마을 막걸리",
            "화담숲주막_참살이 막걸리",
            "화담숲주막_단호박 식혜 ",
            "화담숲주막_찹쌀식혜",
            "화담숲주막_콜라",
            "화담숲주막_스프라이트"
        ],
        "안주": [
            "화담숲주막_해물파전",
            "화담숲주막_병천순대"
        ]
    },
    "화담숲카페": {
        "전통 음료": [
            "화담숲카페_메밀미숫가루",
            "화담숲카페_현미뻥스크림"
        ],
        "커피": [
            "화담숲카페_아메리카노 HOT",
            "화담숲카페_아메리카노 ICE",
            "화담숲카페_카페라떼 ICE"
        ]
    }

}

all_preds_group = []

for store_name, groups in tqdm(menu_groups.items()):
    for group_name, menu_list in groups.items():
        # 그룹 데이터 추출
        x_train_group = x_train[x_train['영업장명_메뉴명'].isin(menu_list)]
        y_train_group = y_train.loc[x_train_group.index]

        if len(x_train_group) == 0:
            continue

        # 카테고리 인코딩
        x_cat_train_group = onehot.fit_transform(x_train_group[categorical_features]).toarray()
        x_num_train_group = x_train_group[numerical_features].values
        x_train_processed_group = np.hstack([x_cat_train_group, x_num_train_group])

        # 그룹별 모델 학습
        model_group = XGBRegressor(**model_params)
        multi_reg_group = MultiOutputRegressor(model_group)
        multi_reg_group.fit(x_train_processed_group, y_train_group)

        # 각 test 파일별 예측
        for path in test_files:
            test_df = pd.read_csv(path)
            x_test, _ = preprocess(test_df, is_train=False)

            x_test_group = x_test[x_test['영업장명_메뉴명'].isin(menu_list)]
            if len(x_test_group) == 0:
                continue

            x_cat_test_group = onehot.transform(x_test_group[categorical_features]).toarray()
            x_num_test_group = x_test_group[numerical_features].values
            x_test_processed_group = np.hstack([x_cat_test_group, x_num_test_group])

            preds_group = multi_reg_group.predict(x_test_processed_group)

            results_group = []
            for i in range(len(x_test_group)):
                menu = x_test_group.iloc[i]['영업장명_메뉴명']
                for j in range(PREDICT):
                    date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                    results_group.append({
                        '영업일자': date,
                        '영업장명_메뉴명': menu,
                        '매출수량': preds_group[i][j],
                        '그룹명': group_name   # 그룹 단위 정보 추가
                    })

            pred_df_group = pd.DataFrame(results_group)
            all_preds_group.append(pred_df_group)

full_pred_df_group = pd.concat(all_preds_group, ignore_index=True)

# === 앙상블 ===
ensemble_preds = all_preds_global.copy()

full_pred_df_local = full_pred_df_local.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_local = full_pred_df_local.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_menu = full_pred_df_menu.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_menu = full_pred_df_menu.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_loc = full_pred_df_loc.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_loc = full_pred_df_loc.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_group = full_pred_df_group.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_group = full_pred_df_group.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

ensemble_preds['매출수량'] = ensemble_preds['매출수량']*(1/5) + full_pred_df_local['매출수량']*(1/5) + full_pred_df_menu['매출수량']*(1/5) + full_pred_df_loc['매출수량']*(1/5) + full_pred_df_group['매출수량']*(1/5)

# 제출 파일 생성
submission = pd.read_csv('sample_submission.csv')
cols = submission.columns.drop('영업일자')
for col in cols:
    df = ensemble_preds[ensemble_preds['영업장명_메뉴명'] == col]
    df = df.set_index('영업일자')
    df = df.loc[submission['영업일자']].reset_index()
    submission[col] = round_and_clip_min1(df['매출수량'])

submission[cols] = submission[cols].clip(lower=1)
submission.to_csv('LGBM_FIN_21011928.csv', index=False, encoding='utf-8-sig')
submission.head(10)

In [ ]:
import os
import random
import glob
import re

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

import itertools
from datetime import timedelta
from tqdm import tqdm

seed = 724
LOOKBACK, PREDICT, BATCH_SIZE, EPOCHS = 28, 7, 16, 50
DROP_COUNT = 17

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(seed)

train = pd.read_csv('train/train.csv')
holiday = pd.to_datetime([
    '2023-01-01', '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24', '2023-03-01', '2023-05-05', '2023-05-27', '2023-05-29', '2023-06-06', '2023-08-15', '2023-09-28', '2023-09-29', '2023-09-30', '2023-10-01', '2023-10-02', '2023-10-03', '2023-10-09', '2023-12-25',
    '2024-01-01', '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12', '2024-03-01', '2024-04-10', '2024-05-06', '2024-05-15', '2024-06-06', '2024-08-15', '2024-09-16', '2024-09-17', '2024-09-18', '2024-10-01', '2024-10-03', '2024-10-09', '2024-12-25',
    '2025-01-01', '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30', '2025-03-03', '2025-05-05', '2025-05-06', '2025-06-03', '2025-06-06', '2025-08-15',
])


def check_holiday(date: pd.Timestamp) -> bool:
    return date.dayofweek in (5, 6) or date in holiday


def filter_long_zero_runs(X, y):
    lag_cols = [f'lag_{i}' for i in range(1, LOOKBACK+1)]
    def _max_zero_run(arr):
        return max((len(list(g)) for v, g in itertools.groupby(arr) if v == 0.0), default=0)
    X['max_zero_run'] = X[lag_cols].apply(_max_zero_run, axis=1)
    mask = X['max_zero_run'] < DROP_COUNT
    return X.loc[mask].drop(columns='max_zero_run'), y.loc[mask]


def interpolate_single_zero_lags(X):
    lag_cols = [f'lag_{i}' for i in range(1, LOOKBACK+1)]
    arr = X[lag_cols].astype(float).to_numpy(copy=True)
    L = arr.shape[1]
    mask = np.zeros_like(arr, dtype=bool)

    for j in range(1, L-1):
        mask[:, j] = (arr[:, j]==0) & (arr[:, j-1]!=0) & (arr[:, j+1]!=0)

    arr[mask] = np.nan
    df_interp = pd.DataFrame(arr, columns=lag_cols, index=X.index).interpolate(axis=1, method='linear').fillna(0)
    X[lag_cols] = df_interp
    return X


def preprocess(df, is_train=True):
    df.loc[df['매출수량'] < 0, '매출수량'] = 0
    df['영업일자'] = pd.to_datetime(df['영업일자'])
    df['dow'] = df['영업일자'].dt.dayofweek
    df['month'] = df['영업일자'].dt.month
    df['holiday'] = df['영업일자'].map(check_holiday).astype(int)

    def make_feature_row(key, ref_date, lag_block):
        data = {
            '영업장명_메뉴명': key,
            'dow': ref_date.dayofweek,
            'month': ref_date.month,
            'holiday': int(check_holiday(ref_date)),
            'sin_day': np.sin(2*np.pi*ref_date.timetuple().tm_yday/365.0),
            'cos_day': np.cos(2*np.pi*ref_date.timetuple().tm_yday/365.0),
            'ref_date': ref_date,
        }
        data.update({f'holiday_plus_{d}d': int(check_holiday(ref_date + pd.Timedelta(days=d))) for d in range(1, 8)})
        data.update({f'lag_{j+1}': float(val) for j, val in enumerate(lag_block)})
        return data


    X_data, y_data = [], []
    for key, group in df.groupby('영업장명_메뉴명'):
        group = group.sort_values('영업일자')
        sales, dates = group['매출수량'].to_numpy(), group['영업일자']

        if len(sales) < LOOKBACK:
            continue

        if is_train:
            Xy = [
                (make_feature_row(key, dates.iloc[i], sales[i-LOOKBACK:i][::-1]), sales[i:i+PREDICT])
                for i in range(LOOKBACK, len(sales) - PREDICT + 1)
            ]
            X_data.extend([r for r, _ in Xy])
            y_data.extend([y for _, y in Xy])
        else:
            ref_date = dates.iloc[-1] + timedelta(days=1)
            lag_block = sales[-LOOKBACK:][::-1]
            X_data.append(make_feature_row(key, ref_date, lag_block))

    X = pd.DataFrame.from_records(X_data)
    y = pd.DataFrame(y_data, columns=[f'target_{i}' for i in range(1, PREDICT+1)]) if is_train else None

    if is_train:
        X, y = filter_long_zero_runs(X, y)

    X = interpolate_single_zero_lags(X)

    # special_stores = {'느티나무 BBQ','라그로타','화담숲주막','화담숲카페'}
    # X['holiday2'] = ((X['영업장명_메뉴명'].str.split('_').str[0].isin(special_stores)) & (X['dow']==0)).astype(int)

    return X, y

def round_and_clip_min1(arr):
    a = np.rint(np.asarray(arr, dtype=float))
    a = np.where(a < 1.0, 1.0, a)
    return a

x_train, y_train = preprocess(train, is_train=True)

from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

onehot = OneHotEncoder(handle_unknown='ignore')

categorical_features = ['영업장명_메뉴명', 'dow', 'month']
numerical_features = ['holiday'] + x_train.filter(regex='holiday_plus').columns.tolist() + ['sin_day', 'cos_day'] + [f'lag_{l}' for l in range(1, LOOKBACK + 1)]

x_cat_train = onehot.fit_transform(x_train[categorical_features]).toarray()
x_num_train = x_train[numerical_features].values
x_train_processed = np.hstack([x_cat_train, x_num_train])

model_params = {
    'n_estimators': 1000,
    'subsample': 0.8,
    'max_depth': 6,
    'colsample_bytree': 0.8,
    'learning_rate': 0.05,
    'objective': 'reg:squarederror',
    'random_state': seed,
}
model = XGBRegressor(**model_params)
multi_output_regressor = MultiOutputRegressor(model)

multi_output_regressor.fit(x_train_processed, y_train)

all_preds = []

test_files = sorted(glob.glob('test/TEST_*.csv'))
for path in test_files:
    results = []

    test_df = pd.read_csv(path)
    x_test, _ = preprocess(test_df, is_train=False)

    x_cat_test = onehot.transform(x_test[categorical_features]).toarray()
    x_num_test = x_test[numerical_features].values
    x_test_processed = np.hstack([x_cat_test, x_num_test])

    preds = multi_output_regressor.predict(x_test_processed)

    for i in range(len(x_test)):
        menu = x_test.iloc[i]['영업장명_메뉴명']

        for j in range(PREDICT):
            date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
            results.append({
                '영업일자': date,
                '영업장명_메뉴명': menu,
                '매출수량': preds[i][j]
            })

    pred_df = pd.DataFrame(results)
    all_preds.append(pred_df)

full_pred_df = pd.concat(all_preds, ignore_index=True)

from collections import defaultdict

# 기존 전체 모델 예측 결과 저장
all_preds_global = full_pred_df.copy()

# 개별 모델 예측 결과 저장
all_preds_local = []

unique_keys = train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).unique()

for key in tqdm(unique_keys):
    x_train_key, y_train_key = x_train[x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key], y_train[x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key]

    # 카테고리 인코딩 (key만 존재하므로 카테고리 의미 없음)
    x_cat_train_key = onehot.fit_transform(x_train_key[categorical_features]).toarray()
    x_num_train_key = x_train_key[numerical_features].values
    x_train_processed_key = np.hstack([x_cat_train_key, x_num_train_key])

    # 개별 모델 학습
    model_key = XGBRegressor(**model_params)
    multi_reg_key = MultiOutputRegressor(model_key)
    multi_reg_key.fit(x_train_processed_key, y_train_key)

    # 각 test 파일별 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)

        x_test_key = x_test[x_test['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key]

        x_cat_test_key = onehot.transform(x_test_key[categorical_features]).toarray()
        x_num_test_key = x_test_key[numerical_features].values
        x_test_processed_key = np.hstack([x_cat_test_key, x_num_test_key])

        preds_key = multi_reg_key.predict(x_test_processed_key)

        results_key = []
        for i in range(len(x_test_key)):
            menu = x_test_key.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_key.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_key[i][j]
                })

        pred_df_key = pd.DataFrame(results_key)
        all_preds_local.append(pred_df_key)

# 개별 모델 전체 결과 합치기
full_pred_df_local = pd.concat(all_preds_local, ignore_index=True)
all_preds_local_menu = []
unique_store_menu = train['영업장명_메뉴명'].unique()

for sm in tqdm(unique_store_menu):
    y_train_sm = y_train[x_train['영업장명_메뉴명'] == sm]
    x_train_sm = x_train[x_train['영업장명_메뉴명'] == sm]

    # 카테고리 / 수치 분리
    x_cat_sm = onehot.fit_transform(x_train_sm[categorical_features]).toarray()
    x_num_sm = x_train_sm[numerical_features].values
    x_train_processed_sm = np.hstack([x_cat_sm, x_num_sm])

    model_sm = XGBRegressor(**model_params)
    multi_reg_sm = MultiOutputRegressor(model_sm)
    multi_reg_sm.fit(x_train_processed_sm, y_train_sm)

    # 테스트 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)
        x_test_sm = x_test[x_test['영업장명_메뉴명'] == sm]

        if len(x_test_sm) == 0:
            continue  # 해당 메뉴 없는 경우 건너뛰기

        x_cat_test_sm = onehot.transform(x_test_sm[categorical_features]).toarray()
        x_num_test_sm = x_test_sm[numerical_features].values
        x_test_processed_sm = np.hstack([x_cat_test_sm, x_num_test_sm])

        preds_sm = multi_reg_sm.predict(x_test_processed_sm)

        results_sm = []
        for i in range(len(x_test_sm)):
            menu = x_test_sm.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_sm.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_sm[i][j]
                })
        all_preds_local_menu.append(pd.DataFrame(results_sm))

full_pred_df_menu = pd.concat(all_preds_local_menu, ignore_index=True)

# group = {
#     0: ['담하', '미라시아', '느티나무 셀프BBQ', '포레스트릿', '카페테리아'],
#     1: ['라그로타'],
#     2: ['연회장'],
#     3: ['화담숲주막', '화담숲카페']
# }

group = {
    0: ['담하', '미라시아', '느티나무 셀프BBQ', '포레스트릿'],
    1: ['라그로타', '카페테리아'],
    2: ['연회장'],
    3: ['화담숲주막', '화담숲카페']
}

all_preds_loc = []

for group_id, store_list in tqdm(group.items()):
    # 해당 그룹에 속하는 train 데이터 선택
    mask_train = x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).isin(store_list)
    x_train_group = x_train[mask_train]
    y_train_group = y_train[mask_train]

    # 카테고리 인코딩
    x_cat_train_group = onehot.fit_transform(x_train_group[categorical_features]).toarray()
    x_num_train_group = x_train_group[numerical_features].values
    x_train_processed_group = np.hstack([x_cat_train_group, x_num_train_group])

    # 그룹별 모델 학습
    model_group = XGBRegressor(**model_params)
    multi_reg_group = MultiOutputRegressor(model_group)
    multi_reg_group.fit(x_train_processed_group, y_train_group)

    # 각 test 파일별 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)

        # 해당 그룹 test 데이터 선택
        mask_test = x_test['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).isin(store_list)
        x_test_group = x_test[mask_test]

        if len(x_test_group) == 0:
            continue  # 해당 그룹 데이터가 없으면 스킵

        x_cat_test_group = onehot.transform(x_test_group[categorical_features]).toarray()
        x_num_test_group = x_test_group[numerical_features].values
        x_test_processed_group = np.hstack([x_cat_test_group, x_num_test_group])

        preds_group = multi_reg_group.predict(x_test_processed_group)

        results_group = []
        for i in range(len(x_test_group)):
            menu = x_test_group.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_group.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_group[i][j]
                })

        pred_df_group = pd.DataFrame(results_group)
        all_preds_loc.append(pred_df_group)

# 그룹별 모델 전체 결과 합치기
full_pred_df_loc = pd.concat(all_preds_loc, ignore_index=True)

menu_groups = {
    "느티나무 셀프BBQ": {
        "메인메뉴": [
            "느티나무 셀프BBQ_BBQ55(단체)",
            "느티나무 셀프BBQ_본삼겹 (단품,실내)",
            "느티나무 셀프BBQ_신라면",
            "느티나무 셀프BBQ_육개장 사발면",
            "느티나무 셀프BBQ_햇반"
        ],
        "사이드": [
            "느티나무 셀프BBQ_쌈야채세트",
            "느티나무 셀프BBQ_쌈장",
            "느티나무 셀프BBQ_허브솔트"
        ],
        "음료": [
            "느티나무 셀프BBQ_참이슬 (단체)",
            "느티나무 셀프BBQ_카스 병(단체)",
            "느티나무 셀프BBQ_콜라 (단체)",
            "느티나무 셀프BBQ_스프라이트 (단체)"
        ],
        "소모품": [
            "느티나무 셀프BBQ_1인 수저세트",
            "느티나무 셀프BBQ_일회용 소주컵",
            "느티나무 셀프BBQ_일회용 종이컵",
            "느티나무 셀프BBQ_친환경 접시 14cm",
            "느티나무 셀프BBQ_친환경 접시 23cm"
        ],
        "대여": [
            "느티나무 셀프BBQ_대여료 30,000원",
            "느티나무 셀프BBQ_대여료 60,000원",
            "느티나무 셀프BBQ_대여료 90,000원",
            "느티나무 셀프BBQ_잔디그늘집 대여료 (6인석)",
            "느티나무 셀프BBQ_잔디그늘집 대여료 (12인석)",
            "느티나무 셀프BBQ_잔디그늘집 의자 추가"
        ]
    },
    "담하": {
        "메인메뉴": [
            "담하_(단체) 생목살 김치전골 2.0",
            "담하_(단체) 은이버섯 갈비탕",
            "담하_(단체) 한우 우거지 국밥",
            "담하_(단체) 황태해장국 3/27까지",
            "담하_(정식) 된장찌개",
            "담하_(정식) 물냉면 ",
            "담하_(정식) 비빔냉면",
            "담하_(후식) 된장찌개",
            "담하_(후식) 물냉면",
            "담하_(후식) 비빔냉면",
            "담하_갑오징어 비빔밥",
            "담하_갱시기",
            "담하_꼬막 비빔밥",
            "담하_담하 한우 불고기",
            "담하_담하 한우 불고기 정식",
            "담하_더덕 한우 지짐",
            "담하_들깨 양지탕",
            "담하_명태회 비빔냉면",
            "담하_봉평메밀 물냉면",
            "담하_생목살 김치찌개",
            "담하_은이버섯 갈비탕",
            "담하_한우 떡갈비 정식",
            "담하_한우 미역국 정식",
            "담하_한우 우거지 국밥",
            "담하_한우 차돌박이 된장찌개",
            "담하_황태해장국"
        ],
        "사이드": [
            "담하_(단체) 공깃밥",
            "담하_공깃밥",
            "담하_라면사리",
            "담하_메밀면 사리"
        ],
        "음료": [
            "담하_느린마을 막걸리",
            "담하_명인안동소주",
            "담하_문막 복분자 칵테일",
            "담하_스프라이트",
            "담하_제로콜라",
            "담하_참이슬",
            "담하_처음처럼",
            "담하_카스",
            "담하_콜라",
            "담하_테라",
            "담하_하동 매실 칵테일"
        ],
        "대여": [
            "담하_룸 이용료"
        ]
    },
    "라그로타": {
        "메인메뉴": [
            "라그로타_AUS (200g)",
            "라그로타_한우 (200g)",
            "라그로타_양갈비 (4ps)",
            "라그로타_까르보나라",
            "라그로타_알리오 에 올리오 ",
            "라그로타_버섯 크림 리조또",
            "라그로타_해산물 토마토 리조또",
            "라그로타_해산물 토마토 스파게티",
            "라그로타_해산물 토마토 스튜 파스타",
            "라그로타_모둠 해산물 플래터"
        ],
        "샐러드·사이드": [
            "라그로타_그릴드 비프 샐러드",
            "라그로타_시저 샐러드 ",
            "라그로타_빵 추가 (1인)",
            "라그로타_Open Food"
        ],
        "음료": [
            "라그로타_아메리카노",
            "라그로타_자몽리치에이드",
            "라그로타_스프라이트",
            "라그로타_제로콜라",
            "라그로타_콜라",
            "라그로타_G-Charge(3)",
            "라그로타_Gls.Sileni",
            "라그로타_Gls.미션 서드",
            "라그로타_미션 서드 카베르네 쉬라",
            "라그로타_카스",
            "라그로타_하이네켄(생)"
        ],
    },
    "미라시아": {
        "브런치·패키지": [
            "미라시아_(단체)브런치주중 36,000",
            "미라시아_미라시아 브런치 (패키지)",
            "미라시아_브런치 2인 패키지 ",
            "미라시아_브런치 4인 패키지 ",
            "미라시아_브런치(대인) 주말",
            "미라시아_브런치(대인) 주중",
            "미라시아_브런치(어린이)"
        ],
        "플래터·피자·파스타": [
            "미라시아_(오븐) 하와이안 쉬림프 피자",
            "미라시아_(화덕) 불고기 페퍼로니 반반피자",
            "미라시아_BBQ Platter",
            "미라시아_BBQ 고기추가",
            "미라시아_보일링 랍스타 플래터",
            "미라시아_보일링 랍스타 플래터(덜매운맛)",
            "미라시아_쉬림프 투움바 파스타",
            "미라시아_오븐구이 윙과 킬바사소세지"
        ],
        "사이드·추가": [
            "미라시아_공깃밥",
            "미라시아_칠리 치즈 프라이",
            "미라시아_파스타면 추가(150g)",
            "미라시아_콥 샐러드"
        ],
        "음료": [
            "미라시아_스프라이트",
            "미라시아_애플망고 에이드",
            "미라시아_핑크레몬에이드",
            "미라시아_코카콜라",
            "미라시아_코카콜라(제로)",
            "미라시아_글라스와인 (레드)",
            "미라시아_레인보우칵테일(알코올)",
            "미라시아_버드와이저(무제한)",
            "미라시아_스텔라(무제한)",
            "미라시아_얼그레이 하이볼",
            "미라시아_유자 하이볼",
            "미라시아_잭 애플 토닉"
        ],
    },
    "연회장": {
        "공간대여": [
            "연회장_Conference L1",
            "연회장_Conference L2",
            "연회장_Conference L3",
            "연회장_Conference M1",
            "연회장_Conference M8",
            "연회장_Conference M9",
            "연회장_Convention Hall",
            "연회장_Grand Ballroom",
            "연회장_OPUS 2"
        ],
        "메인요리": [
            "연회장_돈목살 김치찌개 (밥포함)",
            "연회장_마라샹궈",
            "연회장_매콤 무뼈닭발&계란찜",
            "연회장_모둠 돈육구이(3인)",
            "연회장_왕갈비치킨"
        ],
        "기타": [
            "연회장_공깃밥",
            "연회장_삼겹살추가 (200g)",
            "연회장_야채추가",
            "연회장_주먹밥 (2ea)",
            "연회장_Cass Beer",
            "연회장_Regular Coffee",
            "연회장_Cookie Platter",
            "연회장_골뱅이무침",
            "연회장_로제 치즈떡볶이"
        ],
    },
    "카페테리아": {
        "정식·단체식": [
            "카페테리아_단체식 13000(신)",
            "카페테리아_단체식 18000(신)",
            "카페테리아_한상 삼겹구이 정식(2인) 소요시간 약 15~20분",
            "카페테리아_오픈푸드"
        ],
        "단품메뉴": [
            "카페테리아_돼지고기 김치찌개",
            "카페테리아_수제 등심 돈까스",
            "카페테리아_치즈돈까스",
            "카페테리아_어린이 돈까스",
            "카페테리아_약 고추장 돌솥비빔밥",
            "카페테리아_진사골 설렁탕"
        ],
        "면·밥류": [
            "카페테리아_새우 볶음밥",
            "카페테리아_새우튀김 우동",
            "카페테리아_짜장면",
            "카페테리아_짜장밥",
            "카페테리아_짬뽕",
            "카페테리아_짬뽕밥"
        ],
        "음료": [
            "카페테리아_아메리카노(HOT)",
            "카페테리아_아메리카노(ICE)",
            "카페테리아_카페라떼(HOT)",
            "카페테리아_카페라떼(ICE)",
            "카페테리아_복숭아 아이스티"
        ],
        "사이드·추가": [
            "카페테리아_공깃밥(추가)",
            "카페테리아_샷 추가",
            "카페테리아_구슬아이스크림"
        ]
    },
    "포레스트릿": {
        "분식": [
            "포레스트릿_떡볶이",
            "포레스트릿_꼬치어묵",
            "포레스트릿_치즈 핫도그",
            "포레스트릿_페스츄리 소시지"
        ],
        "음료": [
            "포레스트릿_아메리카노(HOT)",
            "포레스트릿_아메리카노(ICE)",
            "포레스트릿_카페라떼(HOT)",
            "포레스트릿_카페라떼(ICE)",
            "포레스트릿_복숭아 아이스티",
            "포레스트릿_코카콜라",
            "포레스트릿_스프라이트",
            "포레스트릿_생수"
        ]
    },
    "화담숲주막": {
        "음료": [
            "화담숲주막_느린마을 막걸리",
            "화담숲주막_참살이 막걸리",
            "화담숲주막_단호박 식혜 ",
            "화담숲주막_찹쌀식혜",
            "화담숲주막_콜라",
            "화담숲주막_스프라이트"
        ],
        "안주": [
            "화담숲주막_해물파전",
            "화담숲주막_병천순대"
        ]
    },
    "화담숲카페": {
        "전통 음료": [
            "화담숲카페_메밀미숫가루",
            "화담숲카페_현미뻥스크림"
        ],
        "커피": [
            "화담숲카페_아메리카노 HOT",
            "화담숲카페_아메리카노 ICE",
            "화담숲카페_카페라떼 ICE"
        ]
    }

}

all_preds_group = []

for store_name, groups in tqdm(menu_groups.items()):
    for group_name, menu_list in groups.items():
        # 그룹 데이터 추출
        x_train_group = x_train[x_train['영업장명_메뉴명'].isin(menu_list)]
        y_train_group = y_train.loc[x_train_group.index]

        if len(x_train_group) == 0:
            continue

        # 카테고리 인코딩
        x_cat_train_group = onehot.fit_transform(x_train_group[categorical_features]).toarray()
        x_num_train_group = x_train_group[numerical_features].values
        x_train_processed_group = np.hstack([x_cat_train_group, x_num_train_group])

        # 그룹별 모델 학습
        model_group = XGBRegressor(**model_params)
        multi_reg_group = MultiOutputRegressor(model_group)
        multi_reg_group.fit(x_train_processed_group, y_train_group)

        # 각 test 파일별 예측
        for path in test_files:
            test_df = pd.read_csv(path)
            x_test, _ = preprocess(test_df, is_train=False)

            x_test_group = x_test[x_test['영업장명_메뉴명'].isin(menu_list)]
            if len(x_test_group) == 0:
                continue

            x_cat_test_group = onehot.transform(x_test_group[categorical_features]).toarray()
            x_num_test_group = x_test_group[numerical_features].values
            x_test_processed_group = np.hstack([x_cat_test_group, x_num_test_group])

            preds_group = multi_reg_group.predict(x_test_processed_group)

            results_group = []
            for i in range(len(x_test_group)):
                menu = x_test_group.iloc[i]['영업장명_메뉴명']
                for j in range(PREDICT):
                    date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                    results_group.append({
                        '영업일자': date,
                        '영업장명_메뉴명': menu,
                        '매출수량': preds_group[i][j],
                        '그룹명': group_name   # 그룹 단위 정보 추가
                    })

            pred_df_group = pd.DataFrame(results_group)
            all_preds_group.append(pred_df_group)

full_pred_df_group = pd.concat(all_preds_group, ignore_index=True)

# === 앙상블 ===
ensemble_preds = all_preds_global.copy()

full_pred_df_local = full_pred_df_local.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_local = full_pred_df_local.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_menu = full_pred_df_menu.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_menu = full_pred_df_menu.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_loc = full_pred_df_loc.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_loc = full_pred_df_loc.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_group = full_pred_df_group.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_group = full_pred_df_group.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

ensemble_preds['매출수량'] = ensemble_preds['매출수량']*(1/5) + full_pred_df_local['매출수량']*(1/5) + full_pred_df_menu['매출수량']*(1/5) + full_pred_df_loc['매출수량']*(1/5) + full_pred_df_group['매출수량']*(1/5)

# 제출 파일 생성
submission = pd.read_csv('sample_submission.csv')
cols = submission.columns.drop('영업일자')
for col in cols:
    df = ensemble_preds[ensemble_preds['영업장명_메뉴명'] == col]
    df = df.set_index('영업일자')
    df = df.loc[submission['영업일자']].reset_index()
    submission[col] = round_and_clip_min1(df['매출수량'])

submission[cols] = submission[cols].clip(lower=1)
submission.to_csv('LGBM_FIN_724.csv', index=False, encoding='utf-8-sig')
submission.head(10)

In [ ]:
import os
import random
import glob
import re

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

import itertools
from datetime import timedelta
from tqdm import tqdm

seed = 313
LOOKBACK, PREDICT, BATCH_SIZE, EPOCHS = 28, 7, 16, 50
DROP_COUNT = 17

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(seed)

train = pd.read_csv('train/train.csv')
holiday = pd.to_datetime([
    '2023-01-01', '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24', '2023-03-01', '2023-05-05', '2023-05-27', '2023-05-29', '2023-06-06', '2023-08-15', '2023-09-28', '2023-09-29', '2023-09-30', '2023-10-01', '2023-10-02', '2023-10-03', '2023-10-09', '2023-12-25',
    '2024-01-01', '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12', '2024-03-01', '2024-04-10', '2024-05-06', '2024-05-15', '2024-06-06', '2024-08-15', '2024-09-16', '2024-09-17', '2024-09-18', '2024-10-01', '2024-10-03', '2024-10-09', '2024-12-25',
    '2025-01-01', '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30', '2025-03-03', '2025-05-05', '2025-05-06', '2025-06-03', '2025-06-06', '2025-08-15',
])


def check_holiday(date: pd.Timestamp) -> bool:
    return date.dayofweek in (5, 6) or date in holiday


def filter_long_zero_runs(X, y):
    lag_cols = [f'lag_{i}' for i in range(1, LOOKBACK+1)]
    def _max_zero_run(arr):
        return max((len(list(g)) for v, g in itertools.groupby(arr) if v == 0.0), default=0)
    X['max_zero_run'] = X[lag_cols].apply(_max_zero_run, axis=1)
    mask = X['max_zero_run'] < DROP_COUNT
    return X.loc[mask].drop(columns='max_zero_run'), y.loc[mask]


def interpolate_single_zero_lags(X):
    lag_cols = [f'lag_{i}' for i in range(1, LOOKBACK+1)]
    arr = X[lag_cols].astype(float).to_numpy(copy=True)
    L = arr.shape[1]
    mask = np.zeros_like(arr, dtype=bool)

    for j in range(1, L-1):
        mask[:, j] = (arr[:, j]==0) & (arr[:, j-1]!=0) & (arr[:, j+1]!=0)

    arr[mask] = np.nan
    df_interp = pd.DataFrame(arr, columns=lag_cols, index=X.index).interpolate(axis=1, method='linear').fillna(0)
    X[lag_cols] = df_interp
    return X


def preprocess(df, is_train=True):
    df.loc[df['매출수량'] < 0, '매출수량'] = 0
    df['영업일자'] = pd.to_datetime(df['영업일자'])
    df['dow'] = df['영업일자'].dt.dayofweek
    df['month'] = df['영업일자'].dt.month
    df['holiday'] = df['영업일자'].map(check_holiday).astype(int)

    def make_feature_row(key, ref_date, lag_block):
        data = {
            '영업장명_메뉴명': key,
            'dow': ref_date.dayofweek,
            'month': ref_date.month,
            'holiday': int(check_holiday(ref_date)),
            'sin_day': np.sin(2*np.pi*ref_date.timetuple().tm_yday/365.0),
            'cos_day': np.cos(2*np.pi*ref_date.timetuple().tm_yday/365.0),
            'ref_date': ref_date,
        }
        data.update({f'holiday_plus_{d}d': int(check_holiday(ref_date + pd.Timedelta(days=d))) for d in range(1, 8)})
        data.update({f'lag_{j+1}': float(val) for j, val in enumerate(lag_block)})
        return data


    X_data, y_data = [], []
    for key, group in df.groupby('영업장명_메뉴명'):
        group = group.sort_values('영업일자')
        sales, dates = group['매출수량'].to_numpy(), group['영업일자']

        if len(sales) < LOOKBACK:
            continue

        if is_train:
            Xy = [
                (make_feature_row(key, dates.iloc[i], sales[i-LOOKBACK:i][::-1]), sales[i:i+PREDICT])
                for i in range(LOOKBACK, len(sales) - PREDICT + 1)
            ]
            X_data.extend([r for r, _ in Xy])
            y_data.extend([y for _, y in Xy])
        else:
            ref_date = dates.iloc[-1] + timedelta(days=1)
            lag_block = sales[-LOOKBACK:][::-1]
            X_data.append(make_feature_row(key, ref_date, lag_block))

    X = pd.DataFrame.from_records(X_data)
    y = pd.DataFrame(y_data, columns=[f'target_{i}' for i in range(1, PREDICT+1)]) if is_train else None

    if is_train:
        X, y = filter_long_zero_runs(X, y)

    X = interpolate_single_zero_lags(X)

    # special_stores = {'느티나무 BBQ','라그로타','화담숲주막','화담숲카페'}
    # X['holiday2'] = ((X['영업장명_메뉴명'].str.split('_').str[0].isin(special_stores)) & (X['dow']==0)).astype(int)

    return X, y

def round_and_clip_min1(arr):
    a = np.rint(np.asarray(arr, dtype=float))
    a = np.where(a < 1.0, 1.0, a)
    return a

x_train, y_train = preprocess(train, is_train=True)

from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

onehot = OneHotEncoder(handle_unknown='ignore')

categorical_features = ['영업장명_메뉴명', 'dow', 'month']
numerical_features = ['holiday'] + x_train.filter(regex='holiday_plus').columns.tolist() + ['sin_day', 'cos_day'] + [f'lag_{l}' for l in range(1, LOOKBACK + 1)]

x_cat_train = onehot.fit_transform(x_train[categorical_features]).toarray()
x_num_train = x_train[numerical_features].values
x_train_processed = np.hstack([x_cat_train, x_num_train])

model_params = {
    'n_estimators': 1000,
    'subsample': 0.8,
    'max_depth': 6,
    'colsample_bytree': 0.8,
    'learning_rate': 0.05,
    'objective': 'reg:squarederror',
    'random_state': seed,
}
model = XGBRegressor(**model_params)
multi_output_regressor = MultiOutputRegressor(model)

multi_output_regressor.fit(x_train_processed, y_train)

all_preds = []

test_files = sorted(glob.glob('test/TEST_*.csv'))
for path in test_files:
    results = []

    test_df = pd.read_csv(path)
    x_test, _ = preprocess(test_df, is_train=False)

    x_cat_test = onehot.transform(x_test[categorical_features]).toarray()
    x_num_test = x_test[numerical_features].values
    x_test_processed = np.hstack([x_cat_test, x_num_test])

    preds = multi_output_regressor.predict(x_test_processed)

    for i in range(len(x_test)):
        menu = x_test.iloc[i]['영업장명_메뉴명']

        for j in range(PREDICT):
            date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
            results.append({
                '영업일자': date,
                '영업장명_메뉴명': menu,
                '매출수량': preds[i][j]
            })

    pred_df = pd.DataFrame(results)
    all_preds.append(pred_df)

full_pred_df = pd.concat(all_preds, ignore_index=True)

from collections import defaultdict

# 기존 전체 모델 예측 결과 저장
all_preds_global = full_pred_df.copy()

# 개별 모델 예측 결과 저장
all_preds_local = []

unique_keys = train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).unique()

for key in tqdm(unique_keys):
    x_train_key, y_train_key = x_train[x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key], y_train[x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key]

    # 카테고리 인코딩 (key만 존재하므로 카테고리 의미 없음)
    x_cat_train_key = onehot.fit_transform(x_train_key[categorical_features]).toarray()
    x_num_train_key = x_train_key[numerical_features].values
    x_train_processed_key = np.hstack([x_cat_train_key, x_num_train_key])

    # 개별 모델 학습
    model_key = XGBRegressor(**model_params)
    multi_reg_key = MultiOutputRegressor(model_key)
    multi_reg_key.fit(x_train_processed_key, y_train_key)

    # 각 test 파일별 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)

        x_test_key = x_test[x_test['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key]

        x_cat_test_key = onehot.transform(x_test_key[categorical_features]).toarray()
        x_num_test_key = x_test_key[numerical_features].values
        x_test_processed_key = np.hstack([x_cat_test_key, x_num_test_key])

        preds_key = multi_reg_key.predict(x_test_processed_key)

        results_key = []
        for i in range(len(x_test_key)):
            menu = x_test_key.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_key.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_key[i][j]
                })

        pred_df_key = pd.DataFrame(results_key)
        all_preds_local.append(pred_df_key)

# 개별 모델 전체 결과 합치기
full_pred_df_local = pd.concat(all_preds_local, ignore_index=True)
all_preds_local_menu = []
unique_store_menu = train['영업장명_메뉴명'].unique()

for sm in tqdm(unique_store_menu):
    y_train_sm = y_train[x_train['영업장명_메뉴명'] == sm]
    x_train_sm = x_train[x_train['영업장명_메뉴명'] == sm]

    # 카테고리 / 수치 분리
    x_cat_sm = onehot.fit_transform(x_train_sm[categorical_features]).toarray()
    x_num_sm = x_train_sm[numerical_features].values
    x_train_processed_sm = np.hstack([x_cat_sm, x_num_sm])

    model_sm = XGBRegressor(**model_params)
    multi_reg_sm = MultiOutputRegressor(model_sm)
    multi_reg_sm.fit(x_train_processed_sm, y_train_sm)

    # 테스트 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)
        x_test_sm = x_test[x_test['영업장명_메뉴명'] == sm]

        if len(x_test_sm) == 0:
            continue  # 해당 메뉴 없는 경우 건너뛰기

        x_cat_test_sm = onehot.transform(x_test_sm[categorical_features]).toarray()
        x_num_test_sm = x_test_sm[numerical_features].values
        x_test_processed_sm = np.hstack([x_cat_test_sm, x_num_test_sm])

        preds_sm = multi_reg_sm.predict(x_test_processed_sm)

        results_sm = []
        for i in range(len(x_test_sm)):
            menu = x_test_sm.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_sm.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_sm[i][j]
                })
        all_preds_local_menu.append(pd.DataFrame(results_sm))

full_pred_df_menu = pd.concat(all_preds_local_menu, ignore_index=True)

# group = {
#     0: ['담하', '미라시아', '느티나무 셀프BBQ', '포레스트릿', '카페테리아'],
#     1: ['라그로타'],
#     2: ['연회장'],
#     3: ['화담숲주막', '화담숲카페']
# }

group = {
    0: ['담하', '미라시아', '느티나무 셀프BBQ', '포레스트릿'],
    1: ['라그로타', '카페테리아'],
    2: ['연회장'],
    3: ['화담숲주막', '화담숲카페']
}

all_preds_loc = []

for group_id, store_list in tqdm(group.items()):
    # 해당 그룹에 속하는 train 데이터 선택
    mask_train = x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).isin(store_list)
    x_train_group = x_train[mask_train]
    y_train_group = y_train[mask_train]

    # 카테고리 인코딩
    x_cat_train_group = onehot.fit_transform(x_train_group[categorical_features]).toarray()
    x_num_train_group = x_train_group[numerical_features].values
    x_train_processed_group = np.hstack([x_cat_train_group, x_num_train_group])

    # 그룹별 모델 학습
    model_group = XGBRegressor(**model_params)
    multi_reg_group = MultiOutputRegressor(model_group)
    multi_reg_group.fit(x_train_processed_group, y_train_group)

    # 각 test 파일별 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)

        # 해당 그룹 test 데이터 선택
        mask_test = x_test['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).isin(store_list)
        x_test_group = x_test[mask_test]

        if len(x_test_group) == 0:
            continue  # 해당 그룹 데이터가 없으면 스킵

        x_cat_test_group = onehot.transform(x_test_group[categorical_features]).toarray()
        x_num_test_group = x_test_group[numerical_features].values
        x_test_processed_group = np.hstack([x_cat_test_group, x_num_test_group])

        preds_group = multi_reg_group.predict(x_test_processed_group)

        results_group = []
        for i in range(len(x_test_group)):
            menu = x_test_group.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_group.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_group[i][j]
                })

        pred_df_group = pd.DataFrame(results_group)
        all_preds_loc.append(pred_df_group)

# 그룹별 모델 전체 결과 합치기
full_pred_df_loc = pd.concat(all_preds_loc, ignore_index=True)

menu_groups = {
    "느티나무 셀프BBQ": {
        "메인메뉴": [
            "느티나무 셀프BBQ_BBQ55(단체)",
            "느티나무 셀프BBQ_본삼겹 (단품,실내)",
            "느티나무 셀프BBQ_신라면",
            "느티나무 셀프BBQ_육개장 사발면",
            "느티나무 셀프BBQ_햇반"
        ],
        "사이드": [
            "느티나무 셀프BBQ_쌈야채세트",
            "느티나무 셀프BBQ_쌈장",
            "느티나무 셀프BBQ_허브솔트"
        ],
        "음료": [
            "느티나무 셀프BBQ_참이슬 (단체)",
            "느티나무 셀프BBQ_카스 병(단체)",
            "느티나무 셀프BBQ_콜라 (단체)",
            "느티나무 셀프BBQ_스프라이트 (단체)"
        ],
        "소모품": [
            "느티나무 셀프BBQ_1인 수저세트",
            "느티나무 셀프BBQ_일회용 소주컵",
            "느티나무 셀프BBQ_일회용 종이컵",
            "느티나무 셀프BBQ_친환경 접시 14cm",
            "느티나무 셀프BBQ_친환경 접시 23cm"
        ],
        "대여": [
            "느티나무 셀프BBQ_대여료 30,000원",
            "느티나무 셀프BBQ_대여료 60,000원",
            "느티나무 셀프BBQ_대여료 90,000원",
            "느티나무 셀프BBQ_잔디그늘집 대여료 (6인석)",
            "느티나무 셀프BBQ_잔디그늘집 대여료 (12인석)",
            "느티나무 셀프BBQ_잔디그늘집 의자 추가"
        ]
    },
    "담하": {
        "메인메뉴": [
            "담하_(단체) 생목살 김치전골 2.0",
            "담하_(단체) 은이버섯 갈비탕",
            "담하_(단체) 한우 우거지 국밥",
            "담하_(단체) 황태해장국 3/27까지",
            "담하_(정식) 된장찌개",
            "담하_(정식) 물냉면 ",
            "담하_(정식) 비빔냉면",
            "담하_(후식) 된장찌개",
            "담하_(후식) 물냉면",
            "담하_(후식) 비빔냉면",
            "담하_갑오징어 비빔밥",
            "담하_갱시기",
            "담하_꼬막 비빔밥",
            "담하_담하 한우 불고기",
            "담하_담하 한우 불고기 정식",
            "담하_더덕 한우 지짐",
            "담하_들깨 양지탕",
            "담하_명태회 비빔냉면",
            "담하_봉평메밀 물냉면",
            "담하_생목살 김치찌개",
            "담하_은이버섯 갈비탕",
            "담하_한우 떡갈비 정식",
            "담하_한우 미역국 정식",
            "담하_한우 우거지 국밥",
            "담하_한우 차돌박이 된장찌개",
            "담하_황태해장국"
        ],
        "사이드": [
            "담하_(단체) 공깃밥",
            "담하_공깃밥",
            "담하_라면사리",
            "담하_메밀면 사리"
        ],
        "음료": [
            "담하_느린마을 막걸리",
            "담하_명인안동소주",
            "담하_문막 복분자 칵테일",
            "담하_스프라이트",
            "담하_제로콜라",
            "담하_참이슬",
            "담하_처음처럼",
            "담하_카스",
            "담하_콜라",
            "담하_테라",
            "담하_하동 매실 칵테일"
        ],
        "대여": [
            "담하_룸 이용료"
        ]
    },
    "라그로타": {
        "메인메뉴": [
            "라그로타_AUS (200g)",
            "라그로타_한우 (200g)",
            "라그로타_양갈비 (4ps)",
            "라그로타_까르보나라",
            "라그로타_알리오 에 올리오 ",
            "라그로타_버섯 크림 리조또",
            "라그로타_해산물 토마토 리조또",
            "라그로타_해산물 토마토 스파게티",
            "라그로타_해산물 토마토 스튜 파스타",
            "라그로타_모둠 해산물 플래터"
        ],
        "샐러드·사이드": [
            "라그로타_그릴드 비프 샐러드",
            "라그로타_시저 샐러드 ",
            "라그로타_빵 추가 (1인)",
            "라그로타_Open Food"
        ],
        "음료": [
            "라그로타_아메리카노",
            "라그로타_자몽리치에이드",
            "라그로타_스프라이트",
            "라그로타_제로콜라",
            "라그로타_콜라",
            "라그로타_G-Charge(3)",
            "라그로타_Gls.Sileni",
            "라그로타_Gls.미션 서드",
            "라그로타_미션 서드 카베르네 쉬라",
            "라그로타_카스",
            "라그로타_하이네켄(생)"
        ],
    },
    "미라시아": {
        "브런치·패키지": [
            "미라시아_(단체)브런치주중 36,000",
            "미라시아_미라시아 브런치 (패키지)",
            "미라시아_브런치 2인 패키지 ",
            "미라시아_브런치 4인 패키지 ",
            "미라시아_브런치(대인) 주말",
            "미라시아_브런치(대인) 주중",
            "미라시아_브런치(어린이)"
        ],
        "플래터·피자·파스타": [
            "미라시아_(오븐) 하와이안 쉬림프 피자",
            "미라시아_(화덕) 불고기 페퍼로니 반반피자",
            "미라시아_BBQ Platter",
            "미라시아_BBQ 고기추가",
            "미라시아_보일링 랍스타 플래터",
            "미라시아_보일링 랍스타 플래터(덜매운맛)",
            "미라시아_쉬림프 투움바 파스타",
            "미라시아_오븐구이 윙과 킬바사소세지"
        ],
        "사이드·추가": [
            "미라시아_공깃밥",
            "미라시아_칠리 치즈 프라이",
            "미라시아_파스타면 추가(150g)",
            "미라시아_콥 샐러드"
        ],
        "음료": [
            "미라시아_스프라이트",
            "미라시아_애플망고 에이드",
            "미라시아_핑크레몬에이드",
            "미라시아_코카콜라",
            "미라시아_코카콜라(제로)",
            "미라시아_글라스와인 (레드)",
            "미라시아_레인보우칵테일(알코올)",
            "미라시아_버드와이저(무제한)",
            "미라시아_스텔라(무제한)",
            "미라시아_얼그레이 하이볼",
            "미라시아_유자 하이볼",
            "미라시아_잭 애플 토닉"
        ],
    },
    "연회장": {
        "공간대여": [
            "연회장_Conference L1",
            "연회장_Conference L2",
            "연회장_Conference L3",
            "연회장_Conference M1",
            "연회장_Conference M8",
            "연회장_Conference M9",
            "연회장_Convention Hall",
            "연회장_Grand Ballroom",
            "연회장_OPUS 2"
        ],
        "메인요리": [
            "연회장_돈목살 김치찌개 (밥포함)",
            "연회장_마라샹궈",
            "연회장_매콤 무뼈닭발&계란찜",
            "연회장_모둠 돈육구이(3인)",
            "연회장_왕갈비치킨"
        ],
        "기타": [
            "연회장_공깃밥",
            "연회장_삼겹살추가 (200g)",
            "연회장_야채추가",
            "연회장_주먹밥 (2ea)",
            "연회장_Cass Beer",
            "연회장_Regular Coffee",
            "연회장_Cookie Platter",
            "연회장_골뱅이무침",
            "연회장_로제 치즈떡볶이"
        ],
    },
    "카페테리아": {
        "정식·단체식": [
            "카페테리아_단체식 13000(신)",
            "카페테리아_단체식 18000(신)",
            "카페테리아_한상 삼겹구이 정식(2인) 소요시간 약 15~20분",
            "카페테리아_오픈푸드"
        ],
        "단품메뉴": [
            "카페테리아_돼지고기 김치찌개",
            "카페테리아_수제 등심 돈까스",
            "카페테리아_치즈돈까스",
            "카페테리아_어린이 돈까스",
            "카페테리아_약 고추장 돌솥비빔밥",
            "카페테리아_진사골 설렁탕"
        ],
        "면·밥류": [
            "카페테리아_새우 볶음밥",
            "카페테리아_새우튀김 우동",
            "카페테리아_짜장면",
            "카페테리아_짜장밥",
            "카페테리아_짬뽕",
            "카페테리아_짬뽕밥"
        ],
        "음료": [
            "카페테리아_아메리카노(HOT)",
            "카페테리아_아메리카노(ICE)",
            "카페테리아_카페라떼(HOT)",
            "카페테리아_카페라떼(ICE)",
            "카페테리아_복숭아 아이스티"
        ],
        "사이드·추가": [
            "카페테리아_공깃밥(추가)",
            "카페테리아_샷 추가",
            "카페테리아_구슬아이스크림"
        ]
    },
    "포레스트릿": {
        "분식": [
            "포레스트릿_떡볶이",
            "포레스트릿_꼬치어묵",
            "포레스트릿_치즈 핫도그",
            "포레스트릿_페스츄리 소시지"
        ],
        "음료": [
            "포레스트릿_아메리카노(HOT)",
            "포레스트릿_아메리카노(ICE)",
            "포레스트릿_카페라떼(HOT)",
            "포레스트릿_카페라떼(ICE)",
            "포레스트릿_복숭아 아이스티",
            "포레스트릿_코카콜라",
            "포레스트릿_스프라이트",
            "포레스트릿_생수"
        ]
    },
    "화담숲주막": {
        "음료": [
            "화담숲주막_느린마을 막걸리",
            "화담숲주막_참살이 막걸리",
            "화담숲주막_단호박 식혜 ",
            "화담숲주막_찹쌀식혜",
            "화담숲주막_콜라",
            "화담숲주막_스프라이트"
        ],
        "안주": [
            "화담숲주막_해물파전",
            "화담숲주막_병천순대"
        ]
    },
    "화담숲카페": {
        "전통 음료": [
            "화담숲카페_메밀미숫가루",
            "화담숲카페_현미뻥스크림"
        ],
        "커피": [
            "화담숲카페_아메리카노 HOT",
            "화담숲카페_아메리카노 ICE",
            "화담숲카페_카페라떼 ICE"
        ]
    }

}

all_preds_group = []

for store_name, groups in tqdm(menu_groups.items()):
    for group_name, menu_list in groups.items():
        # 그룹 데이터 추출
        x_train_group = x_train[x_train['영업장명_메뉴명'].isin(menu_list)]
        y_train_group = y_train.loc[x_train_group.index]

        if len(x_train_group) == 0:
            continue

        # 카테고리 인코딩
        x_cat_train_group = onehot.fit_transform(x_train_group[categorical_features]).toarray()
        x_num_train_group = x_train_group[numerical_features].values
        x_train_processed_group = np.hstack([x_cat_train_group, x_num_train_group])

        # 그룹별 모델 학습
        model_group = XGBRegressor(**model_params)
        multi_reg_group = MultiOutputRegressor(model_group)
        multi_reg_group.fit(x_train_processed_group, y_train_group)

        # 각 test 파일별 예측
        for path in test_files:
            test_df = pd.read_csv(path)
            x_test, _ = preprocess(test_df, is_train=False)

            x_test_group = x_test[x_test['영업장명_메뉴명'].isin(menu_list)]
            if len(x_test_group) == 0:
                continue

            x_cat_test_group = onehot.transform(x_test_group[categorical_features]).toarray()
            x_num_test_group = x_test_group[numerical_features].values
            x_test_processed_group = np.hstack([x_cat_test_group, x_num_test_group])

            preds_group = multi_reg_group.predict(x_test_processed_group)

            results_group = []
            for i in range(len(x_test_group)):
                menu = x_test_group.iloc[i]['영업장명_메뉴명']
                for j in range(PREDICT):
                    date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                    results_group.append({
                        '영업일자': date,
                        '영업장명_메뉴명': menu,
                        '매출수량': preds_group[i][j],
                        '그룹명': group_name   # 그룹 단위 정보 추가
                    })

            pred_df_group = pd.DataFrame(results_group)
            all_preds_group.append(pred_df_group)

full_pred_df_group = pd.concat(all_preds_group, ignore_index=True)

# === 앙상블 ===
ensemble_preds = all_preds_global.copy()

full_pred_df_local = full_pred_df_local.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_local = full_pred_df_local.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_menu = full_pred_df_menu.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_menu = full_pred_df_menu.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_loc = full_pred_df_loc.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_loc = full_pred_df_loc.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_group = full_pred_df_group.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_group = full_pred_df_group.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

ensemble_preds['매출수량'] = ensemble_preds['매출수량']*(1/5) + full_pred_df_local['매출수량']*(1/5) + full_pred_df_menu['매출수량']*(1/5) + full_pred_df_loc['매출수량']*(1/5) + full_pred_df_group['매출수량']*(1/5)

# 제출 파일 생성
submission = pd.read_csv('sample_submission.csv')
cols = submission.columns.drop('영업일자')
for col in cols:
    df = ensemble_preds[ensemble_preds['영업장명_메뉴명'] == col]
    df = df.set_index('영업일자')
    df = df.loc[submission['영업일자']].reset_index()
    submission[col] = round_and_clip_min1(df['매출수량'])

submission[cols] = submission[cols].clip(lower=1)
submission.to_csv('LGBM_FIN_313.csv', index=False, encoding='utf-8-sig')
submission.head(10)

In [ ]:
import os
import random
import glob
import re

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

import itertools
from datetime import timedelta
from tqdm import tqdm

seed = 42
LOOKBACK, PREDICT, BATCH_SIZE, EPOCHS = 28, 7, 16, 50
DROP_COUNT = 17

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(seed)

train = pd.read_csv('train/train.csv')
holiday = pd.to_datetime([
    '2023-01-01', '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24', '2023-03-01', '2023-05-05', '2023-05-27', '2023-05-29', '2023-06-06', '2023-08-15', '2023-09-28', '2023-09-29', '2023-09-30', '2023-10-01', '2023-10-02', '2023-10-03', '2023-10-09', '2023-12-25',
    '2024-01-01', '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12', '2024-03-01', '2024-04-10', '2024-05-06', '2024-05-15', '2024-06-06', '2024-08-15', '2024-09-16', '2024-09-17', '2024-09-18', '2024-10-01', '2024-10-03', '2024-10-09', '2024-12-25',
    '2025-01-01', '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30', '2025-03-03', '2025-05-05', '2025-05-06', '2025-06-03', '2025-06-06', '2025-08-15',
])


def check_holiday(date: pd.Timestamp) -> bool:
    return date.dayofweek in (5, 6) or date in holiday


def filter_long_zero_runs(X, y):
    lag_cols = [f'lag_{i}' for i in range(1, LOOKBACK+1)]
    def _max_zero_run(arr):
        return max((len(list(g)) for v, g in itertools.groupby(arr) if v == 0.0), default=0)
    X['max_zero_run'] = X[lag_cols].apply(_max_zero_run, axis=1)
    mask = X['max_zero_run'] < DROP_COUNT
    return X.loc[mask].drop(columns='max_zero_run'), y.loc[mask]


def interpolate_single_zero_lags(X):
    lag_cols = [f'lag_{i}' for i in range(1, LOOKBACK+1)]
    arr = X[lag_cols].astype(float).to_numpy(copy=True)
    L = arr.shape[1]
    mask = np.zeros_like(arr, dtype=bool)

    for j in range(1, L-1):
        mask[:, j] = (arr[:, j]==0) & (arr[:, j-1]!=0) & (arr[:, j+1]!=0)

    arr[mask] = np.nan
    df_interp = pd.DataFrame(arr, columns=lag_cols, index=X.index).interpolate(axis=1, method='linear').fillna(0)
    X[lag_cols] = df_interp
    return X


def preprocess(df, is_train=True):
    df.loc[df['매출수량'] < 0, '매출수량'] = 0
    df['영업일자'] = pd.to_datetime(df['영업일자'])
    df['dow'] = df['영업일자'].dt.dayofweek
    df['month'] = df['영업일자'].dt.month
    df['holiday'] = df['영업일자'].map(check_holiday).astype(int)

    def make_feature_row(key, ref_date, lag_block):
        data = {
            '영업장명_메뉴명': key,
            'dow': ref_date.dayofweek,
            'month': ref_date.month,
            'holiday': int(check_holiday(ref_date)),
            'sin_day': np.sin(2*np.pi*ref_date.timetuple().tm_yday/365.0),
            'cos_day': np.cos(2*np.pi*ref_date.timetuple().tm_yday/365.0),
            'ref_date': ref_date,
        }
        data.update({f'holiday_plus_{d}d': int(check_holiday(ref_date + pd.Timedelta(days=d))) for d in range(1, 8)})
        data.update({f'lag_{j+1}': float(val) for j, val in enumerate(lag_block)})
        return data


    X_data, y_data = [], []
    for key, group in df.groupby('영업장명_메뉴명'):
        group = group.sort_values('영업일자')
        sales, dates = group['매출수량'].to_numpy(), group['영업일자']

        if len(sales) < LOOKBACK:
            continue

        if is_train:
            Xy = [
                (make_feature_row(key, dates.iloc[i], sales[i-LOOKBACK:i][::-1]), sales[i:i+PREDICT])
                for i in range(LOOKBACK, len(sales) - PREDICT + 1)
            ]
            X_data.extend([r for r, _ in Xy])
            y_data.extend([y for _, y in Xy])
        else:
            ref_date = dates.iloc[-1] + timedelta(days=1)
            lag_block = sales[-LOOKBACK:][::-1]
            X_data.append(make_feature_row(key, ref_date, lag_block))

    X = pd.DataFrame.from_records(X_data)
    y = pd.DataFrame(y_data, columns=[f'target_{i}' for i in range(1, PREDICT+1)]) if is_train else None

    if is_train:
        X, y = filter_long_zero_runs(X, y)

    X = interpolate_single_zero_lags(X)

    # special_stores = {'느티나무 BBQ','라그로타','화담숲주막','화담숲카페'}
    # X['holiday2'] = ((X['영업장명_메뉴명'].str.split('_').str[0].isin(special_stores)) & (X['dow']==0)).astype(int)

    return X, y

def round_and_clip_min1(arr):
    a = np.rint(np.asarray(arr, dtype=float))
    a = np.where(a < 1.0, 1.0, a)
    return a

x_train, y_train = preprocess(train, is_train=True)

from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

onehot = OneHotEncoder(handle_unknown='ignore')

categorical_features = ['영업장명_메뉴명', 'dow', 'month']
numerical_features = ['holiday'] + x_train.filter(regex='holiday_plus').columns.tolist() + ['sin_day', 'cos_day'] + [f'lag_{l}' for l in range(1, LOOKBACK + 1)]

x_cat_train = onehot.fit_transform(x_train[categorical_features]).toarray()
x_num_train = x_train[numerical_features].values
x_train_processed = np.hstack([x_cat_train, x_num_train])

model_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': seed,
}
model = XGBRegressor(**model_params)
multi_output_regressor = MultiOutputRegressor(model)
multi_output_regressor.fit(x_train_processed, y_train)

all_preds = []

test_files = sorted(glob.glob('test/TEST_*.csv'))
for path in test_files:
    results = []

    test_df = pd.read_csv(path)
    x_test, _ = preprocess(test_df, is_train=False)

    x_cat_test = onehot.transform(x_test[categorical_features]).toarray()
    x_num_test = x_test[numerical_features].values
    x_test_processed = np.hstack([x_cat_test, x_num_test])

    preds = multi_output_regressor.predict(x_test_processed)

    for i in range(len(x_test)):
        menu = x_test.iloc[i]['영업장명_메뉴명']

        for j in range(PREDICT):
            date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
            results.append({
                '영업일자': date,
                '영업장명_메뉴명': menu,
                '매출수량': preds[i][j]
            })

    pred_df = pd.DataFrame(results)
    all_preds.append(pred_df)

full_pred_df = pd.concat(all_preds, ignore_index=True)

from collections import defaultdict

# 기존 전체 모델 예측 결과 저장
all_preds_global = full_pred_df.copy()

# 개별 모델 예측 결과 저장
all_preds_local = []

unique_keys = train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).unique()

for key in tqdm(unique_keys):
    x_train_key, y_train_key = x_train[x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key], y_train[x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key]

    # 카테고리 인코딩 (key만 존재하므로 카테고리 의미 없음)
    x_cat_train_key = onehot.fit_transform(x_train_key[categorical_features]).toarray()
    x_num_train_key = x_train_key[numerical_features].values
    x_train_processed_key = np.hstack([x_cat_train_key, x_num_train_key])

    # 개별 모델 학습
    model_key = XGBRegressor(**model_params)
    multi_reg_key = MultiOutputRegressor(model_key)
    multi_reg_key.fit(x_train_processed_key, y_train_key)

    # 각 test 파일별 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)

        x_test_key = x_test[x_test['영업장명_메뉴명'].map(lambda x: x.split('_')[0]) == key]

        x_cat_test_key = onehot.transform(x_test_key[categorical_features]).toarray()
        x_num_test_key = x_test_key[numerical_features].values
        x_test_processed_key = np.hstack([x_cat_test_key, x_num_test_key])

        preds_key = multi_reg_key.predict(x_test_processed_key)

        results_key = []
        for i in range(len(x_test_key)):
            menu = x_test_key.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_key.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_key[i][j]
                })

        pred_df_key = pd.DataFrame(results_key)
        all_preds_local.append(pred_df_key)

# 개별 모델 전체 결과 합치기
full_pred_df_local = pd.concat(all_preds_local, ignore_index=True)


all_preds_local_menu = []
unique_store_menu = train['영업장명_메뉴명'].unique()

for sm in tqdm(unique_store_menu):
    y_train_sm = y_train[x_train['영업장명_메뉴명'] == sm]
    x_train_sm = x_train[x_train['영업장명_메뉴명'] == sm]

    # 카테고리 / 수치 분리
    x_cat_sm = onehot.fit_transform(x_train_sm[categorical_features]).toarray()
    x_num_sm = x_train_sm[numerical_features].values
    x_train_processed_sm = np.hstack([x_cat_sm, x_num_sm])

    model_sm = XGBRegressor(**model_params)
    multi_reg_sm = MultiOutputRegressor(model_sm)
    multi_reg_sm.fit(x_train_processed_sm, y_train_sm)

    # 테스트 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)
        x_test_sm = x_test[x_test['영업장명_메뉴명'] == sm]

        if len(x_test_sm) == 0:
            continue  # 해당 메뉴 없는 경우 건너뛰기

        x_cat_test_sm = onehot.transform(x_test_sm[categorical_features]).toarray()
        x_num_test_sm = x_test_sm[numerical_features].values
        x_test_processed_sm = np.hstack([x_cat_test_sm, x_num_test_sm])

        preds_sm = multi_reg_sm.predict(x_test_processed_sm)

        results_sm = []
        for i in range(len(x_test_sm)):
            menu = x_test_sm.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_sm.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_sm[i][j]
                })
        all_preds_local_menu.append(pd.DataFrame(results_sm))

full_pred_df_menu = pd.concat(all_preds_local_menu, ignore_index=True)

# group = {
#     0: ['담하', '미라시아', '느티나무 셀프BBQ', '포레스트릿', '카페테리아'],
#     1: ['라그로타'],
#     2: ['연회장'],
#     3: ['화담숲주막', '화담숲카페']
# }

group = {
    0: ['담하', '미라시아', '느티나무 셀프BBQ', '포레스트릿'],
    1: ['라그로타', '카페테리아'],
    2: ['연회장'],
    3: ['화담숲주막', '화담숲카페']
}

all_preds_loc = []

for group_id, store_list in tqdm(group.items()):
    # 해당 그룹에 속하는 train 데이터 선택
    mask_train = x_train['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).isin(store_list)
    x_train_group = x_train[mask_train]
    y_train_group = y_train[mask_train]

    # 카테고리 인코딩
    x_cat_train_group = onehot.fit_transform(x_train_group[categorical_features]).toarray()
    x_num_train_group = x_train_group[numerical_features].values
    x_train_processed_group = np.hstack([x_cat_train_group, x_num_train_group])

    # 그룹별 모델 학습
    model_group = XGBRegressor(**model_params)
    multi_reg_group = MultiOutputRegressor(model_group)
    multi_reg_group.fit(x_train_processed_group, y_train_group)

    # 각 test 파일별 예측
    for path in test_files:
        test_df = pd.read_csv(path)
        x_test, _ = preprocess(test_df, is_train=False)

        # 해당 그룹 test 데이터 선택
        mask_test = x_test['영업장명_메뉴명'].map(lambda x: x.split('_')[0]).isin(store_list)
        x_test_group = x_test[mask_test]

        if len(x_test_group) == 0:
            continue  # 해당 그룹 데이터가 없으면 스킵

        x_cat_test_group = onehot.transform(x_test_group[categorical_features]).toarray()
        x_num_test_group = x_test_group[numerical_features].values
        x_test_processed_group = np.hstack([x_cat_test_group, x_num_test_group])

        preds_group = multi_reg_group.predict(x_test_processed_group)

        results_group = []
        for i in range(len(x_test_group)):
            menu = x_test_group.iloc[i]['영업장명_메뉴명']
            for j in range(PREDICT):
                date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                results_group.append({
                    '영업일자': date,
                    '영업장명_메뉴명': menu,
                    '매출수량': preds_group[i][j]
                })

        pred_df_group = pd.DataFrame(results_group)
        all_preds_loc.append(pred_df_group)

# 그룹별 모델 전체 결과 합치기
full_pred_df_loc = pd.concat(all_preds_loc, ignore_index=True)

menu_groups = {
    "느티나무 셀프BBQ": {
        "메인메뉴": [
            "느티나무 셀프BBQ_BBQ55(단체)",
            "느티나무 셀프BBQ_본삼겹 (단품,실내)",
            "느티나무 셀프BBQ_신라면",
            "느티나무 셀프BBQ_육개장 사발면",
            "느티나무 셀프BBQ_햇반"
        ],
        "사이드": [
            "느티나무 셀프BBQ_쌈야채세트",
            "느티나무 셀프BBQ_쌈장",
            "느티나무 셀프BBQ_허브솔트"
        ],
        "음료": [
            "느티나무 셀프BBQ_참이슬 (단체)",
            "느티나무 셀프BBQ_카스 병(단체)",
            "느티나무 셀프BBQ_콜라 (단체)",
            "느티나무 셀프BBQ_스프라이트 (단체)"
        ],
        "소모품": [
            "느티나무 셀프BBQ_1인 수저세트",
            "느티나무 셀프BBQ_일회용 소주컵",
            "느티나무 셀프BBQ_일회용 종이컵",
            "느티나무 셀프BBQ_친환경 접시 14cm",
            "느티나무 셀프BBQ_친환경 접시 23cm"
        ],
        "대여": [
            "느티나무 셀프BBQ_대여료 30,000원",
            "느티나무 셀프BBQ_대여료 60,000원",
            "느티나무 셀프BBQ_대여료 90,000원",
            "느티나무 셀프BBQ_잔디그늘집 대여료 (6인석)",
            "느티나무 셀프BBQ_잔디그늘집 대여료 (12인석)",
            "느티나무 셀프BBQ_잔디그늘집 의자 추가"
        ]
    },
    "담하": {
        "메인메뉴": [
            "담하_(단체) 생목살 김치전골 2.0",
            "담하_(단체) 은이버섯 갈비탕",
            "담하_(단체) 한우 우거지 국밥",
            "담하_(단체) 황태해장국 3/27까지",
            "담하_(정식) 된장찌개",
            "담하_(정식) 물냉면 ",
            "담하_(정식) 비빔냉면",
            "담하_(후식) 된장찌개",
            "담하_(후식) 물냉면",
            "담하_(후식) 비빔냉면",
            "담하_갑오징어 비빔밥",
            "담하_갱시기",
            "담하_꼬막 비빔밥",
            "담하_담하 한우 불고기",
            "담하_담하 한우 불고기 정식",
            "담하_더덕 한우 지짐",
            "담하_들깨 양지탕",
            "담하_명태회 비빔냉면",
            "담하_봉평메밀 물냉면",
            "담하_생목살 김치찌개",
            "담하_은이버섯 갈비탕",
            "담하_한우 떡갈비 정식",
            "담하_한우 미역국 정식",
            "담하_한우 우거지 국밥",
            "담하_한우 차돌박이 된장찌개",
            "담하_황태해장국"
        ],
        "사이드": [
            "담하_(단체) 공깃밥",
            "담하_공깃밥",
            "담하_라면사리",
            "담하_메밀면 사리"
        ],
        "음료": [
            "담하_느린마을 막걸리",
            "담하_명인안동소주",
            "담하_문막 복분자 칵테일",
            "담하_스프라이트",
            "담하_제로콜라",
            "담하_참이슬",
            "담하_처음처럼",
            "담하_카스",
            "담하_콜라",
            "담하_테라",
            "담하_하동 매실 칵테일"
        ],
        "대여": [
            "담하_룸 이용료"
        ]
    },
    "라그로타": {
        "메인메뉴": [
            "라그로타_AUS (200g)",
            "라그로타_한우 (200g)",
            "라그로타_양갈비 (4ps)",
            "라그로타_까르보나라",
            "라그로타_알리오 에 올리오 ",
            "라그로타_버섯 크림 리조또",
            "라그로타_해산물 토마토 리조또",
            "라그로타_해산물 토마토 스파게티",
            "라그로타_해산물 토마토 스튜 파스타",
            "라그로타_모둠 해산물 플래터"
        ],
        "샐러드·사이드": [
            "라그로타_그릴드 비프 샐러드",
            "라그로타_시저 샐러드 ",
            "라그로타_빵 추가 (1인)",
            "라그로타_Open Food"
        ],
        "음료": [
            "라그로타_아메리카노",
            "라그로타_자몽리치에이드",
            "라그로타_스프라이트",
            "라그로타_제로콜라",
            "라그로타_콜라",
            "라그로타_G-Charge(3)",
            "라그로타_Gls.Sileni",
            "라그로타_Gls.미션 서드",
            "라그로타_미션 서드 카베르네 쉬라",
            "라그로타_카스",
            "라그로타_하이네켄(생)"
        ],
    },
    "미라시아": {
        "브런치·패키지": [
            "미라시아_(단체)브런치주중 36,000",
            "미라시아_미라시아 브런치 (패키지)",
            "미라시아_브런치 2인 패키지 ",
            "미라시아_브런치 4인 패키지 ",
            "미라시아_브런치(대인) 주말",
            "미라시아_브런치(대인) 주중",
            "미라시아_브런치(어린이)"
        ],
        "플래터·피자·파스타": [
            "미라시아_(오븐) 하와이안 쉬림프 피자",
            "미라시아_(화덕) 불고기 페퍼로니 반반피자",
            "미라시아_BBQ Platter",
            "미라시아_BBQ 고기추가",
            "미라시아_보일링 랍스타 플래터",
            "미라시아_보일링 랍스타 플래터(덜매운맛)",
            "미라시아_쉬림프 투움바 파스타",
            "미라시아_오븐구이 윙과 킬바사소세지"
        ],
        "사이드·추가": [
            "미라시아_공깃밥",
            "미라시아_칠리 치즈 프라이",
            "미라시아_파스타면 추가(150g)",
            "미라시아_콥 샐러드"
        ],
        "음료": [
            "미라시아_스프라이트",
            "미라시아_애플망고 에이드",
            "미라시아_핑크레몬에이드",
            "미라시아_코카콜라",
            "미라시아_코카콜라(제로)",
            "미라시아_글라스와인 (레드)",
            "미라시아_레인보우칵테일(알코올)",
            "미라시아_버드와이저(무제한)",
            "미라시아_스텔라(무제한)",
            "미라시아_얼그레이 하이볼",
            "미라시아_유자 하이볼",
            "미라시아_잭 애플 토닉"
        ],
    },
    "연회장": {
        "공간대여": [
            "연회장_Conference L1",
            "연회장_Conference L2",
            "연회장_Conference L3",
            "연회장_Conference M1",
            "연회장_Conference M8",
            "연회장_Conference M9",
            "연회장_Convention Hall",
            "연회장_Grand Ballroom",
            "연회장_OPUS 2"
        ],
        "메인요리": [
            "연회장_돈목살 김치찌개 (밥포함)",
            "연회장_마라샹궈",
            "연회장_매콤 무뼈닭발&계란찜",
            "연회장_모둠 돈육구이(3인)",
            "연회장_왕갈비치킨"
        ],
        "기타": [
            "연회장_공깃밥",
            "연회장_삼겹살추가 (200g)",
            "연회장_야채추가",
            "연회장_주먹밥 (2ea)",
            "연회장_Cass Beer",
            "연회장_Regular Coffee",
            "연회장_Cookie Platter",
            "연회장_골뱅이무침",
            "연회장_로제 치즈떡볶이"
        ],
    },
    "카페테리아": {
        "정식·단체식": [
            "카페테리아_단체식 13000(신)",
            "카페테리아_단체식 18000(신)",
            "카페테리아_한상 삼겹구이 정식(2인) 소요시간 약 15~20분",
            "카페테리아_오픈푸드"
        ],
        "단품메뉴": [
            "카페테리아_돼지고기 김치찌개",
            "카페테리아_수제 등심 돈까스",
            "카페테리아_치즈돈까스",
            "카페테리아_어린이 돈까스",
            "카페테리아_약 고추장 돌솥비빔밥",
            "카페테리아_진사골 설렁탕"
        ],
        "면·밥류": [
            "카페테리아_새우 볶음밥",
            "카페테리아_새우튀김 우동",
            "카페테리아_짜장면",
            "카페테리아_짜장밥",
            "카페테리아_짬뽕",
            "카페테리아_짬뽕밥"
        ],
        "음료": [
            "카페테리아_아메리카노(HOT)",
            "카페테리아_아메리카노(ICE)",
            "카페테리아_카페라떼(HOT)",
            "카페테리아_카페라떼(ICE)",
            "카페테리아_복숭아 아이스티"
        ],
        "사이드·추가": [
            "카페테리아_공깃밥(추가)",
            "카페테리아_샷 추가",
            "카페테리아_구슬아이스크림"
        ]
    },
    "포레스트릿": {
        "분식": [
            "포레스트릿_떡볶이",
            "포레스트릿_꼬치어묵",
            "포레스트릿_치즈 핫도그",
            "포레스트릿_페스츄리 소시지"
        ],
        "음료": [
            "포레스트릿_아메리카노(HOT)",
            "포레스트릿_아메리카노(ICE)",
            "포레스트릿_카페라떼(HOT)",
            "포레스트릿_카페라떼(ICE)",
            "포레스트릿_복숭아 아이스티",
            "포레스트릿_코카콜라",
            "포레스트릿_스프라이트",
            "포레스트릿_생수"
        ]
    },
    "화담숲주막": {
        "음료": [
            "화담숲주막_느린마을 막걸리",
            "화담숲주막_참살이 막걸리",
            "화담숲주막_단호박 식혜 ",
            "화담숲주막_찹쌀식혜",
            "화담숲주막_콜라",
            "화담숲주막_스프라이트"
        ],
        "안주": [
            "화담숲주막_해물파전",
            "화담숲주막_병천순대"
        ]
    },
    "화담숲카페": {
        "전통 음료": [
            "화담숲카페_메밀미숫가루",
            "화담숲카페_현미뻥스크림"
        ],
        "커피": [
            "화담숲카페_아메리카노 HOT",
            "화담숲카페_아메리카노 ICE",
            "화담숲카페_카페라떼 ICE"
        ]
    }

}

all_preds_group = []

for store_name, groups in tqdm(menu_groups.items()):
    for group_name, menu_list in groups.items():
        # 그룹 데이터 추출
        x_train_group = x_train[x_train['영업장명_메뉴명'].isin(menu_list)]
        y_train_group = y_train.loc[x_train_group.index]

        if len(x_train_group) == 0:
            continue

        # 카테고리 인코딩
        x_cat_train_group = onehot.fit_transform(x_train_group[categorical_features]).toarray()
        x_num_train_group = x_train_group[numerical_features].values
        x_train_processed_group = np.hstack([x_cat_train_group, x_num_train_group])

        # 그룹별 모델 학습
        model_group = XGBRegressor(**model_params)
        multi_reg_group = MultiOutputRegressor(model_group)
        multi_reg_group.fit(x_train_processed_group, y_train_group)

        # 각 test 파일별 예측
        for path in test_files:
            test_df = pd.read_csv(path)
            x_test, _ = preprocess(test_df, is_train=False)

            x_test_group = x_test[x_test['영업장명_메뉴명'].isin(menu_list)]
            if len(x_test_group) == 0:
                continue

            x_cat_test_group = onehot.transform(x_test_group[categorical_features]).toarray()
            x_num_test_group = x_test_group[numerical_features].values
            x_test_processed_group = np.hstack([x_cat_test_group, x_num_test_group])

            preds_group = multi_reg_group.predict(x_test_processed_group)

            results_group = []
            for i in range(len(x_test_group)):
                menu = x_test_group.iloc[i]['영업장명_메뉴명']
                for j in range(PREDICT):
                    date = f"{os.path.basename(path).split('.')[0]}+{j+1}일"
                    results_group.append({
                        '영업일자': date,
                        '영업장명_메뉴명': menu,
                        '매출수량': preds_group[i][j],
                        '그룹명': group_name   # 그룹 단위 정보 추가
                    })

            pred_df_group = pd.DataFrame(results_group)
            all_preds_group.append(pred_df_group)

full_pred_df_group = pd.concat(all_preds_group, ignore_index=True)


# === 앙상블 ===
ensemble_preds = all_preds_global.copy()

full_pred_df_local = full_pred_df_local.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_local = full_pred_df_local.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_menu = full_pred_df_menu.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_menu = full_pred_df_menu.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_loc = full_pred_df_loc.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_loc = full_pred_df_loc.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

full_pred_df_group = full_pred_df_group.set_index(['영업일자', '영업장명_메뉴명'])
full_pred_df_group = full_pred_df_group.loc[ensemble_preds.set_index(['영업일자', '영업장명_메뉴명']).index].reset_index()

ensemble_preds['매출수량'] = ensemble_preds['매출수량']*(1/5) + full_pred_df_local['매출수량']*(1/5) + full_pred_df_menu['매출수량']*(1/5) + full_pred_df_loc['매출수량']*(1/5) + full_pred_df_group['매출수량']*(1/5)

# 제출 파일 생성
submission = pd.read_csv('sample_submission.csv')
cols = submission.columns.drop('영업일자')
for col in cols:
    df = ensemble_preds[ensemble_preds['영업장명_메뉴명'] == col]
    df = df.set_index('영업일자')
    df = df.loc[submission['영업일자']].reset_index()
    submission[col] = round_and_clip_min1(df['매출수량'])

submission[cols] = submission[cols].clip(lower=1)
submission.to_csv('submit_ensemble.csv', index=False, encoding='utf-8-sig')
submission.head(10)

In [ ]:
import pandas as pd
files = [
    "LGBM_FIN_427.csv",
    "LGBM_FIN_777.csv",
    "LGBM_FIN_21011928.csv",
    "LGBM_FIN_724.csv",
    "LGBM_FIN_313.csv",
    "submit_ensemble.csv"
]

dfs = [pd.read_csv(file).set_index("영업일자") for file in files]

ensemble_df = sum(dfs) / len(dfs)

ensemble_df = ensemble_df.round(0).astype(int)

ensemble_df = ensemble_df.reset_index()

ensemble_df.to_csv('FINAL_see.csv',index=False)